# Education Services: Overview, Quality, Accessibility and Operations

This notebook examines education and care services using the cleaned service records and supporting geographic and birth-registration data. Run the code cells in their existing order from the project folder.

## Contents

| Section | Focus |
|---|---|
| [1. Setup](#section-1) | Packages and shared chart colours |
| [2. Service Records](#section-2) | Load records and check service identifiers |
| [3. SOS Boundaries](#section-3) | Load the geographic boundaries |
| [4. Data Preparation](#section-4) | Coordinates, spatial matching, location classification and missing fields |
| [5. Supply Overview](#section-5) | Approvals, growth, geographic distribution and estimated capacity |
| [6. Service Quality](#section-6) | Overall Quality, QA1–QA7, state benchmarks and service types |
| [7. Accessibility](#section-7) | Public transport distances by state and location |
| [8. Estimated Demand and Accessibility](#section-8) | Estimated children per service and nearest public transport distance |
| [9. Operational Trends](#section-9) | Activities, combinations and capacity distributions |

**Reading the notebook.** Each section keeps its preparation cells before its figures. Method notes appear beside the relevant analysis. Existing figure outputs and interactive controls are retained.

**Samples and interpretation.** Date restrictions and missing-value rules are specific to each analysis. Supply and demand comparisons use the relevant approval cohorts; current quality and transport comparisons do not impose a common end-of-2024 restriction. Operations distinguishes the full activity-profile sample from the end-of-2024 capacity sample. Refer to the local method notes for each denominator.

**Display.** Charts and results are displayed directly in this notebook. The service data are pandas/GeoPandas tables, not a connected SQL database.


<a id="section-1"></a>

## 1. Setup

`pandas` handles tables, `geopandas` handles maps and coordinates, and `plotly.express` creates interactive charts.


In [ ]:
from matplotlib.ticker import StrMethodFormatter  # Format large counts with thousands separators.
from matplotlib.colors import LinearSegmentedColormap
import plotly.io as pio
import matplotlib.pyplot as plt
import plotly.express as px
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
import json
import numpy as np
# Use the reference chart's named blue/orange palette throughout.
from matplotlib.colors import to_hex
chart_colors = ['tab:blue', 'tab:orange']
location_order = ['Rural', 'Urban']
location_colors = dict(zip(location_order, chart_colors))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=chart_colors)
# Plotly needs CSS colours; convert the same named palette automatically.
plotly_blue, plotly_orange = [to_hex(color) for color in chart_colors]
px.defaults.color_discrete_sequence = [plotly_blue, plotly_orange]


<a id="section-2"></a>

## 2. Load Service Records

The service approval number identifies each service when merging data. First, check for missing and duplicate IDs.

Do not remove every row with a missing value: a missing rating or phone number does not make the entire service record unusable.


In [ ]:
education_raw = pd.read_csv(
    'Education-services-with-station-access_loc.csv',
    dtype={'Postcode': 'string'},
    low_memory=False
)

print('Number of institutions:', len(education_raw))
print('Missing service approval numbers:', education_raw['ServiceApprovalNumber'].isna().sum())
print('Missing service approval numbers:', education_raw['ServiceApprovalNumber'].isnull().sum())
print('Duplicate service approval numbers:', education_raw['ServiceApprovalNumber'].duplicated().sum())

<a id="section-3"></a>

## 3. Load SOS Boundaries


In [ ]:
sos = gpd.read_file('SOS_2021_AUST_GDA2020.shp')
# Table display hidden; calculations are retained.
print("SOS CRS:", sos.crs)
sos = sos.dropna(subset=['geometry'])

<a id="section-4"></a>

## 4. Data Preparation

### 4.1 Extract longitude and latitude from the original coordinates

The original `geometry` values look like `c(151.20, -33.87)`.

Extract the two numbers inside the brackets: the first is longitude and the second is latitude. Values that cannot be converted to numbers remain missing.


In [ ]:
# Extract longitude and latitude from the original c(...) coordinate strings.
Coordinates = education_raw['geometry'].str.extract(
    r'c\(([-\d.]+),\s*([-\d.]+)\)'
)

education_raw['longitude'] = pd.to_numeric(Coordinates[0], errors='coerce')
education_raw['latitude'] = pd.to_numeric(Coordinates[1], errors='coerce')

# Table display hidden; calculations are retained.

# Create a GeoDataFrame from the service coordinates.( assume that education database has a 7844 coordinate system)
education_geo = gpd.GeoDataFrame(
    education_raw,
    geometry=gpd.points_from_xy(
        education_raw['longitude'],
        education_raw['latitude']
    ),
    crs='EPSG:7844'
)

# Check the coordinate reference systems of the two datasets.
print("Education CRS:", education_geo.crs)
print("SOS CRS:", sos.crs)

### 4.2 Join services to SOS areas

Use the original SOS boundaries to attach an area name to each service point. Check unmatched coordinates before classifying Urban and Rural.


In [ ]:
education_sos = gpd.sjoin(
    education_geo,
    sos[['SOS_NAME21',  'geometry']],
    how='left',
    predicate='within'
)
# get rid of the extra column.
education_sos = education_sos.drop(columns=['index_right'])
print(education_sos['SOS_NAME21'].unique())


### 4.3 Validate service coordinates

The records with positive latitude are inconsistent with the Australian locations in this dataset, which are in the Southern Hemisphere. These coordinates are invalid for this analysis, so the following cell excludes them by retaining records with negative latitude.


In [ ]:
education_sos = education_sos[
    education_sos['latitude'] < 0
]

# Table display hidden; calculations are retained.


### 4.4 Classify locations as Urban / Rural using SOS names

Use the following mapping of ABS SOS categories:

- **Major Urban**: Urban
- **Other Urban**: Urban
- **Bounded Locality**: Rural
- **Rural Balance**: Rural

Other or unmatched SOS values are grouped as `Rural` for this analysis. This is an analytical grouping assumption; the original `SOS_NAME21` values are retained.


In [ ]:
urban_rural = {
    'Major Urban': 'Urban',
    'Other Urban': 'Urban',
    'Bounded Locality': 'Rural',
    'Rural Balance': 'Rural'
}

# Map SOS categories into Urban / Rural.
# Group previously unclassified or unmatched SOS records as Rural for this analysis.
education_sos['Classification'] = (
    education_sos['SOS_NAME21']
    .map(urban_rural)
    .fillna('Rural')
)
# Table display hidden; calculations are retained.
education_sos.head()

### 4.5 Review and Remove High-Missingness Fields


In [ ]:
# =========================================================
# Missing Value Treatment
# =========================================================

# Calculate missing rate for each variable
missing_rate = education_sos.isna().mean()

# Identify variables with more than 75% missing values
high_missing_cols = missing_rate[
    missing_rate > 0.75
].index.tolist()

print(f'Number of columns before cleaning: {education_sos.shape[1]}')
print(f'Columns with >75% missing values: {len(high_missing_cols)}')

for col in high_missing_cols:
    print(f'{col}: {missing_rate[col]:.1%}')

In [ ]:
# Drop variables with more than 75% missing values
education_sos = education_sos.drop(
    columns=high_missing_cols
)

print(f'Number of columns after cleaning: {education_sos.shape[1]}')

### 4.6 Remove Fields Not Used in the Analysis


In [ ]:
# drop columns that are not useful for this analysis 
drop_cols = [
    'Address',
    'Phone',
    'Fax',
    'ConditionsOnApproval',
    'FullAddress',
    'suburb',
    'ServiceName',
    'ProviderLegalName',
    'ClosestBusStation',
    'ClosestrainStation'
]

education_sos = education_sos.drop(
    columns=drop_cols,
    errors='ignore'
)
print(f'Number of columns after cleaning: {education_sos.shape[1]}')

<a id="section-5"></a>

## 5. Supply Overview

This section establishes the national supply baseline and compares states and Urban/Rural locations. Estimated demand and its relationship with capacity are presented in Section 8.

| Subsection | Comparison |
|---|---|
| 5.1 National Supply Growth | Cumulative approvals, state distribution and annual new approvals |
| 5.2 State Growth–Volume Matrix | Growth in cumulative approvals and the number of new approvals |
| 5.3 Urban/Rural Distribution | Location shares of the overall sample and 2024 new approvals |
| 5.4 Service Capacity | Estimated approved places nationally, by state and by location |


### 5.1 National Supply Growth

First, compare cumulative approvals at the end of 2023 and 2024 and calculate their percentage change. Then plot annual newly approved services for 2019–2024, with a dashed line showing the 2019–2023 average. Cumulative approval growth and annual new approvals are different measures; neither reconstructs net changes in operating services after closures.


#### 5.1.1 Cumulative Approvals: 2023 and 2024


In [ ]:
# 8.1.1 Cumulative approval comparison: 2023 vs 2024

# Convert approval dates using the original date format.
education_sos['ApprovalDate'] = pd.to_datetime(
    education_sos['ServiceApprovalGrantedDate'],
    format='%d/%m/%Y',
    errors='coerce'
)

# Count unique services approved by the end of each year.
total_2023 = education_sos.loc[
    education_sos['ApprovalDate'] <= '2023-12-31',
    'ServiceApprovalNumber'
].nunique()

total_2024 = education_sos.loc[
    education_sos['ApprovalDate'] <= '2024-12-31',
    'ServiceApprovalNumber'
].nunique()

# Calculate the percentage change.
growth_rate_2024 = (
    (total_2024 - total_2023) / total_2023 * 100
    if total_2023 > 0 else float('nan')
)

print(f"2023 cumulative approvals: {total_2023:,}")
print(f"2024 cumulative approvals: {total_2024:,}")
print(f"Cumulative approval growth: {growth_rate_2024:.2f}%")

#### 5.1.2 State Distribution: Counts and Map


In [ ]:
# Count services approved by the end of 2024
state_counts = (
    education_sos.loc[education_sos['ApprovalDate'] <= '2024-12-31']
    .groupby('State')['ServiceApprovalNumber'].nunique()
    .sort_values().rename('Service_Count').to_frame()
)
state_counts['Share'] = (
    state_counts['Service_Count'] / state_counts['Service_Count'].sum() * 100
)

# Load and match state boundaries
state_names = {
    'New South Wales': 'NSW', 'Victoria': 'VIC', 'Queensland': 'QLD',
    'South Australia': 'SA', 'Western Australia': 'WA', 'Tasmania': 'TAS',
    'Northern Territory': 'NT', 'Australian Capital Territory': 'ACT'
}
states_gdf = gpd.read_file('australia_states.geojson')
states_gdf['State'] = states_gdf['STATE_NAME'].map(state_names)
map_data = states_gdf.join(state_counts, on='State')

In [ ]:
# Sort from smallest to largest
state_bar_data = state_counts.sort_values('Service_Count')
fig, ax = plt.subplots(figsize=(10, 6))
# Horizontal bar chart
bars = ax.barh(
    state_bar_data.index,
    state_bar_data['Service_Count'],
    alpha=0.7,
    height=0.5
)
# Add value labels
ax.bar_label(
    bars,
    labels=[f'{x:,}' for x in state_bar_data['Service_Count']],
    fontsize=10
)
# Title and axis labels
ax.set_title(
    'Approved Services by State — End of 2024',
    fontsize=14
)
ax.set_xlabel('Approved Services')
ax.set_ylabel('')
# Leave space for labels
ax.set_xlim(
    0,
    state_bar_data['Service_Count'].max() * 1.15
)
# Remove gridlines
ax.grid(False)
# Remove unnecessary borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Merge state geometry with service share
map_data = (
    states_gdf.to_crs(7844)
    .set_index('State')
    .join(state_counts)
)
# Create label positions
points = map_data.geometry.representative_point()
fig = go.Figure(go.Choropleth(
    geojson=json.loads(map_data.to_json()),
    locations=map_data.index,
    z=map_data['Share'],
    # One colour with different intensity
    colorscale='Blues',
    marker_line_color='white',
    marker_line_width=1,
    showscale=False,
    hovertemplate='%{location}: %{z:.1f}%<extra></extra>'
))
# Add state labels
fig.add_trace(go.Scattergeo(
    lon=points.x + (map_data.index == 'ACT') * 1.5,
    lat=points.y,
    text=[
        f'{state}<br>{share:.1f}%'
        for state, share in map_data['Share'].items()
    ],
    mode='text',
    textfont=dict(size=15,color=['white' if state in ['NSW', 'VIC'] else 'black'for state in map_data.index]),
    textposition=[
        'middle right' if state == 'ACT' else 'middle center'
        for state in map_data.index
    ],
    hoverinfo='skip'
))
fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='mercator'
)
fig.update_layout(
    title=dict(
        text='Share of Approved Services by State — End of 2024',
        x=0.5
    ),
    width=1000,
    height=600,
    margin=dict(l=0, r=0, t=60, b=0),
    showlegend=False
)
fig.show()

#### 5.1.3 Annual New Approvals: 2019–2024


In [ ]:
# Calculate annual new approvals from 2019 to 2024
annual_counts = (
    education_sos[
        education_sos['ApprovalDate'].dt.year.between(2019, 2024)
    ]
    .groupby(education_sos['ApprovalDate'].dt.year)['ServiceApprovalNumber']
    .nunique()
)
# Rename index
annual_counts.index.name = 'ApprovalYear'
# Calculate the average annual approvals from 2019 to 2023
five_year_average = annual_counts.loc[2019:2023].mean()
# Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(
    annual_counts.index,
    annual_counts.values,
    width=0.3,
    alpha=0.7
)
# 5-year average line
plt.axhline(
    y=five_year_average,
    linestyle='--',
    linewidth=1.5,
    color='tab:orange',
)
# Put average label on the left
plt.text(
    2019,
    five_year_average,
    f'5-Year Average: {five_year_average}',
    ha='left',
    va='bottom'
)
# Add values above each bar
plt.bar_label(bars)
plt.xlabel('Approval Year')
plt.ylabel('Newly Approved Services')
plt.title('Annual New Service Approvals, 2019 - 2024')
plt.xticks(annual_counts.index)
# Remove top and right borders
ax = plt.gca()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

### 5.2 State Growth–Volume Matrix

Use the cleaned `education_sos` table from Sections 1–4 and the approval dates and national totals calculated in Section 5.1. Growth is the percentage increase in cumulative approvals from the end of 2023 to the end of 2024. Volume counts unique services approved during 2024. The horizontal benchmark is the mean new-approval count across states and territories.


In [ ]:
# Use the cleaned service records and approval dates from Sections 1-8.1.
# Count each service approval number once within each state.
state_growth = pd.DataFrame({
    'Total_2023': education_sos.loc[
        education_sos['ApprovalDate'] <= '2023-12-31'
    ].groupby('State')['ServiceApprovalNumber'].nunique(),
    'Total_2024': education_sos.loc[
        education_sos['ApprovalDate'] <= '2024-12-31'
    ].groupby('State')['ServiceApprovalNumber'].nunique(),
    'New_2024': education_sos.loc[
        education_sos['ApprovalDate'].between('2024-01-01', '2024-12-31')
    ].groupby('State')['ServiceApprovalNumber'].nunique()
}).fillna(0).astype(int)

# Match Section 8.1: growth in cumulative approvals, not annual-approval YoY.
state_growth['Growth_Rate_2024'] = (
    (state_growth['Total_2024'] - state_growth['Total_2023'])/state_growth['Total_2023'].replace(0, float('nan')) * 100
)

national_new_2024 = education_sos.loc[
    education_sos['ApprovalDate'].between('2024-01-01', '2024-12-31'),
    'ServiceApprovalNumber'
].nunique()

state_growth['Share_2024'] = (state_growth['New_2024'] / national_new_2024 * 100)
# Reuse the national totals
national_growth_rate = ((total_2024 - total_2023) / total_2023 * 100)
average_volume = state_growth['New_2024'].mean()


In [ ]:
# 1. Prepare plotting data
plot_data = state_growth.reset_index()
# 2. Create state labels
plot_data['Label'] = (
    plot_data['State']
    + '<br>('
    + plot_data['Growth_Rate_2024'].map('{:.1f}'.format)
    + '%, '
    + plot_data['New_2024'].map('{:,.0f}'.format)
    + ')'
)
# 3. Create scatter plot
fig = px.scatter(
    plot_data,
    x='Growth_Rate_2024',
    y='New_2024',
    text='Label',
    hover_name='State',
    hover_data={
        'Label': False,
        'Growth_Rate_2024': ':.1f',
        'New_2024': ':,.0f',
        'Share_2024': ':.1f',
        'Total_2023': ':,.0f',
        'Total_2024': ':,.0f'
    }
)
fig.update_traces(
    marker=dict(size=12, color=plotly_blue, opacity=0.7),
    textposition='top center'
)
# 4. Add national growth benchmark with vertical text
fig.add_vline(
    x=national_growth_rate,
    line_dash='dash',
    line_width=1.5,
    annotation_text=f'National Growth: {national_growth_rate:.1f}%',
    annotation_position='top right',
    annotation_textangle=-90,
    annotation_yanchor='top',
    annotation_xanchor='left',
    annotation_xshift=5
)
# 5. Add average new approvals benchmark
fig.add_hline(
    y=average_volume,
    line_dash='dash',
    line_width=1.5,
    annotation_text=f'Average New Approvals: {average_volume:,.1f}',
    annotation_position='top right'
)
# 6. Configure layout
fig.update_layout(
    width=1000,
    height=600,
    title={
        'text': 'State Growth-Volume Matrix, 2024',
        'x': 0.5
    },
    xaxis_title='2024 Supply Growth (%)',
    yaxis_title='2024 New Approved Services',
    plot_bgcolor='white',
    showlegend=False
)
fig.update_xaxes(
    showgrid=False,
    zeroline=False
)
fig.update_yaxes(
    range=[0, 260],
    showgrid=False,
    zeroline=False
)
fig.show()

### 5.3 Urban/Rural Distribution

Compare the Urban / Rural shares of all services approved by the end of 2024 with those newly approved during 2024. Each column uses its own sample as the denominator; previously unclassified SOS records are included in Rural, following Section 4.4.


In [ ]:
# Overall approved services by the end of 2024
overall_2024 = education_sos[education_sos['ApprovalDate'] <= '2024-12-31'].copy()
# Newly approved services in 2024
new_2024 = education_sos[education_sos['ApprovalDate'].dt.year == 2024].copy()
# Calculate Urban–Rural shares for overall sample
overall_urban_rural = (overall_2024['Classification']
    .value_counts(normalize=True)
    .mul(100)
)
# Calculate Urban–Rural shares for 2024 new approvals
new_urban_rural = (
    new_2024['Classification']
    .value_counts(normalize=True)
    .mul(100)
)
# Combine
urban_rural_comparison = pd.DataFrame({
    'Overall': overall_urban_rural,
    '2024 New': new_urban_rural
}).fillna(0)

urban_rural_comparison

> No further investigation of this distribution is included in this subsection.


### 5.4 Service Capacity

Missing capacities are filled using the observed state mean (national mean if unavailable), keeping original values unchanged. Most missing values are FDC services, whereas state means mainly reflect CBC: totals may be biased, and mean imputation understates variability. Both approval cohorts use the same 2024-based estimates and current capacity records; growth does not reconstruct historical closures or capacity changes.


In [ ]:
# Fill missing capacity using state means from the end-of-2024 sample.
capacity_original = education_sos['NumberOfApprovedPlaces'].copy()
capacity_numeric = pd.to_numeric(capacity_original, errors='coerce')
capacity_snapshot = education_sos['ApprovalDate'] <= '2024-12-31'
capacity_observed = capacity_numeric.loc[capacity_snapshot]
capacity_state_means = capacity_observed.groupby(
    education_sos.loc[capacity_snapshot, 'State']
).mean()
capacity_fill = education_sos['State'].map(capacity_state_means).fillna(capacity_observed.mean())
education_sos['ApprovedPlaces_Clean'] = capacity_numeric.fillna(capacity_fill)
education_sos['ApprovedPlaces_Imputed'] = capacity_original.isna().astype(int)

# Compare cumulative approval cohorts using the same cleaned capacities.
overall_2023 = education_sos[education_sos['ApprovalDate'] <= '2023-12-31'].copy()
overall_2024 = education_sos[education_sos['ApprovalDate'] <= '2024-12-31'].copy()

capacity_2023 = overall_2023['ApprovedPlaces_Clean'].sum()
capacity_2024 = overall_2024['ApprovedPlaces_Clean'].sum()
capacity_growth_pct = (capacity_2024 / capacity_2023 - 1) * 100
national_total_capacity = capacity_2024

# Keep validation in memory, without displaying diagnostic tables.
capacity_validation = pd.DataFrame({
    'Before': capacity_observed.describe(),
    'After': overall_2024['ApprovedPlaces_Clean'].describe()
})

print(f'2023 estimated approved places: {capacity_2023:,.2f}')
print(f'2024 estimated approved places: {capacity_2024:,.2f}')
print(f'Capacity growth: {capacity_growth_pct:.2f}%')


#### 5.4.1 Capacity by State and Urban/Rural Location

Bars show estimated approved places; labels include each group's national share. Previously unclassified SOS records are included in Rural, following Section 4.4.


In [ ]:
# Summarise the same 2024 sample for both charts.
capacity_cross_sections = {}

for column in ['State', 'Classification']:
    table = overall_2024.groupby(column, dropna=False).agg(
        TotalApprovedCapacity=('ApprovedPlaces_Clean', 'sum'),
        Services=('ServiceApprovalNumber', 'nunique')
    )
    table['AveragePlacesPerService'] = (
        table['TotalApprovedCapacity'] / table['Services']
    )
    table['NationalCapacitySharePct'] = (
        table['TotalApprovedCapacity'] / capacity_2024 * 100
    )
    capacity_cross_sections[column] = table.sort_values(
        'TotalApprovedCapacity'
    )

state_capacity = capacity_cross_sections['State']
urban_rural_capacity = (
    capacity_cross_sections['Classification']
    .reindex(location_order)
)

# 1 row × 2 columns
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 5),
    gridspec_kw={'width_ratios': [1.5, 1]}
)

for ax, table, title in zip(
    axes,
    [state_capacity, urban_rural_capacity],
    ['Capacity by State', 'Capacity by Urban / Rural Location']
):
    colors = [
        location_colors.get(group, chart_colors[0])
        for group in table.index
    ]

    bars = ax.barh(
        table.index.fillna('Unclassified'),
        table['TotalApprovedCapacity'],
        height=0.55,
        color=colors,
        alpha=0.7
    )

    if title == 'Capacity by Urban / Rural Location':
        ax.invert_yaxis()

    labels = [
        f'{v:,.0f} ({p:.1f}%)'
        for v, p in zip(
            table['TotalApprovedCapacity'],
            table['NationalCapacitySharePct']
        )
    ]

    ax.bar_label(
        bars,
        labels=labels,
        padding=5,
        fontsize=12
    )

    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlabel('Estimated approved places', fontsize=12)
    ax.set_ylabel('')
    ax.tick_params(labelsize=12)

    ax.set_xlim(
        0,
        table['TotalApprovedCapacity'].max() * 1.5
    )

    ax.xaxis.set_major_formatter(
        StrMethodFormatter('{x:,.0f}')
    )

    ax.locator_params(axis='x', nbins=4)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(False)

fig.suptitle(
    'Estimated Service Capacity - End of 2024',
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.93], w_pad=4)
plt.show()

<a id="section-6"></a>

## 6. Service Quality


Each quality topic has four figures using the original horizontal bar style:

1. Australia's current rating distribution.
2. Current state percentages benchmarked against Australia, by rating level.
3. Current rating distributions by ServiceType: Centre-Based Care and Family Day Care.
4. Australia's previous/current rating distribution on the same paired sample.

Current distributions include all services with a recognised current rating for that topic, without requiring a previous rating. State percentages use the same rule within each state; the national benchmark pools all eligible services. Previous/current comparisons retain the original template's paired non-missing sample and denominator. Unassessed values outside the five rating categories are not plotted in that paired comparison. This is a comparison of available ratings, not a common-year trend.

Service-type comparisons use the current recognised ratings within each type, independently for each topic. Percentages sum to 100% within each type; the legend gives the rated sample size. Differences are descriptive and do not adjust for state mix or other service characteristics.


In [ ]:
rating_order = [
    'Significant Improvement Required',
    'Working Towards NQS',
    'Meeting NQS',
    'Exceeding NQS',
    'Excellent'
]


In [ ]:
quality_state_order = ['NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']
quality_current_results = {}


In [ ]:
quality_service_type_order = ['Centre-Based Care', 'Family Day Care']
quality_service_type_results = {}


### 6.1 Overall Quality

#### 6.1.1 Current Ratings: Australia


In [ ]:
quality_topic = 'Overall Quality'
quality_current_column = 'OverallRating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['overall'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.1.2 Current Ratings: States vs Australia

One figure compares each rating level across states in a 1 x 5 layout, matching the other quality topics. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; state labels include current rated sample sizes.


In [ ]:
# Five rating panels in one row, matching the other quality topics.
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.1.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['overall'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.1.4 Current Overall Quality and Capacity

Compare approved places across the five ordered Overall Quality categories. Use current Centre-Based Care records with a recognised overall rating and observed positive capacity, without requiring a previous rating. This follows the existing capacity-analysis rule of excluding Family Day Care and using no imputed capacities; the sample is current, without the Operations section's end-of-2024 approval restriction.

Each service appears once. Boxes show the median and interquartile range, whiskers extend to 1.5 times the IQR, and all outliers remain visible. Labels show sample sizes. The chart describes the association between rating and capacity; it does not establish causation.


In [ ]:
# Use original observed capacity rather than the imputed capacity field.
overall_quality_capacity = education_sos.loc[
    education_sos['ServiceType'].eq('Centre-Based Care')
    & education_sos['OverallRating'].isin(rating_order),
    ['ServiceApprovalNumber', 'OverallRating', 'NumberOfApprovedPlaces']
].copy()
overall_quality_capacity['ApprovedPlaces'] = pd.to_numeric(
    overall_quality_capacity['NumberOfApprovedPlaces'], errors='coerce'
)
overall_quality_capacity = overall_quality_capacity.loc[
    overall_quality_capacity['ApprovedPlaces'] > 0
]

quality_capacity_groups = [
    overall_quality_capacity.loc[
        overall_quality_capacity['OverallRating'] == rating, 'ApprovedPlaces'
    ]
    for rating in rating_order
]
quality_capacity_labels = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    quality_capacity_groups,
    tick_labels=[
        f'{label}\n(n={len(group):,})'
        for label, group in zip(quality_capacity_labels, quality_capacity_groups)
    ],
    flierprops={'markersize': 2, 'alpha': 0.3},
    medianprops={'color': 'tab:orange'}
)
ax.set_xlabel('Overall Quality Rating')
ax.set_ylabel('Approved Places')
ax.set_title('CBC Capacity by Current Overall Quality Rating')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 6.1.5 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_quality = education_sos.dropna(subset=['PreviousOverallRating','OverallRating'])
print(f'Number of Australian services: {len(australia_quality):,}')

In [ ]:
#Previous vs Current Overall Rating in Australia

# Previous rating distribution
australia_quality_previous_rating = (
    australia_quality['PreviousOverallRating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_quality_current_rating = (
    australia_quality['OverallRating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_quality_previous_rating,
    'Current': australia_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('Overall Rating')
plt.title('Previous vs Current Overall Quality Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.2 QA1: Educational Program and Practice

#### 6.2.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA1'
quality_current_column = 'QualityArea1Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa1'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.2.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.2.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa1'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.2.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa1_quality = education_sos.dropna(subset=['PreviousQualityArea1Rating','QualityArea1Rating'])
print(f'Number of Australian services: {len(australia_qa1_quality):,}')

In [ ]:
#Previous vs Current QA1 Rating in Australia

# Previous rating distribution
australia_qa1_quality_previous_rating = (
    australia_qa1_quality['PreviousQualityArea1Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa1_quality_current_rating = (
    australia_qa1_quality['QualityArea1Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa1_quality_previous_rating,
    'Current': australia_qa1_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA1 Rating')
plt.title('Previous vs Current QA1: Educational Program and Practice Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.3 QA2: Children's Health and Safety

#### 6.3.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA2'
quality_current_column = 'QualityArea2Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa2'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.3.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.3.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa2'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.3.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa2_quality = education_sos.dropna(subset=['PreviousQualityArea2Rating','QualityArea2Rating'])
print(f'Number of Australian services: {len(australia_qa2_quality):,}')

In [ ]:
#Previous vs Current QA2 Rating in Australia

# Previous rating distribution
australia_qa2_quality_previous_rating = (
    australia_qa2_quality['PreviousQualityArea2Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa2_quality_current_rating = (
    australia_qa2_quality['QualityArea2Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa2_quality_previous_rating,
    'Current': australia_qa2_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA2 Rating')
plt.title("Previous vs Current QA2: Children's Health and Safety Ratings in Australia")

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.4 QA3: Physical Environment

#### 6.4.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA3'
quality_current_column = 'QualityArea3Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa3'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.4.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.4.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa3'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.4.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa3_quality = education_sos.dropna(subset=['PreviousQualityArea3Rating','QualityArea3Rating'])
print(f'Number of Australian services: {len(australia_qa3_quality):,}')

In [ ]:
#Previous vs Current QA3 Rating in Australia

# Previous rating distribution
australia_qa3_quality_previous_rating = (
    australia_qa3_quality['PreviousQualityArea3Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa3_quality_current_rating = (
    australia_qa3_quality['QualityArea3Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa3_quality_previous_rating,
    'Current': australia_qa3_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA3 Rating')
plt.title('Previous vs Current QA3: Physical Environment Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.5 QA4: Staffing Arrangements

#### 6.5.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA4'
quality_current_column = 'QualityArea4Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa4'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.5.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.5.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa4'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.5.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa4_quality = education_sos.dropna(subset=['PreviousQualityArea4Rating','QualityArea4Rating'])
print(f'Number of Australian services: {len(australia_qa4_quality):,}')

In [ ]:
#Previous vs Current QA4 Rating in Australia

# Previous rating distribution
australia_qa4_quality_previous_rating = (
    australia_qa4_quality['PreviousQualityArea4Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa4_quality_current_rating = (
    australia_qa4_quality['QualityArea4Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa4_quality_previous_rating,
    'Current': australia_qa4_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA4 Rating')
plt.title('Previous vs Current QA4: Staffing Arrangements Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.6 QA5: Relationships with Children

#### 6.6.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA5'
quality_current_column = 'QualityArea5Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa5'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.6.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.6.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa5'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.6.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa5_quality = education_sos.dropna(subset=['PreviousQualityArea5Rating','QualityArea5Rating'])
print(f'Number of Australian services: {len(australia_qa5_quality):,}')

In [ ]:
#Previous vs Current QA5 Rating in Australia

# Previous rating distribution
australia_qa5_quality_previous_rating = (
    australia_qa5_quality['PreviousQualityArea5Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa5_quality_current_rating = (
    australia_qa5_quality['QualityArea5Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa5_quality_previous_rating,
    'Current': australia_qa5_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA5 Rating')
plt.title('Previous vs Current QA5: Relationships with Children Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.7 QA6: Collaborative Partnerships

#### 6.7.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA6'
quality_current_column = 'QualityArea6Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa6'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.7.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.7.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa6'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.7.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa6_quality = education_sos.dropna(subset=['PreviousQualityArea6Rating','QualityArea6Rating'])
print(f'Number of Australian services: {len(australia_qa6_quality):,}')

In [ ]:
#Previous vs Current QA6 Rating in Australia

# Previous rating distribution
australia_qa6_quality_previous_rating = (
    australia_qa6_quality['PreviousQualityArea6Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa6_quality_current_rating = (
    australia_qa6_quality['QualityArea6Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa6_quality_previous_rating,
    'Current': australia_qa6_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA6 Rating')
plt.title('Previous vs Current QA6: Collaborative Partnerships Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.8 QA7: Governance and Leadership

#### 6.8.1 Current Ratings: Australia


In [ ]:
quality_topic = 'QA7'
quality_current_column = 'QualityArea7Rating'

# Current performance does not require a previous rating.
quality_current = education_sos.loc[
    education_sos[quality_current_column].isin(rating_order)
]
national_current_rating = (
    quality_current[quality_current_column]
    .value_counts(normalize=True)
    .reindex(rating_order, fill_value=0)
    * 100
)

state_current_rating = pd.DataFrame({
    state: quality_current.loc[
        quality_current['State'].eq(state), quality_current_column
    ].value_counts(normalize=True).reindex(rating_order, fill_value=0) * 100
    for state in quality_state_order
}).T
state_current_n = quality_current.groupby('State').size().reindex(
    quality_state_order, fill_value=0
)
# An empty state sample has no percentage estimate.
state_current_rating.loc[state_current_n.eq(0), :] = np.nan
quality_current_results['qa7'] = {
    'Australia': national_current_rating.copy(),
    'State': state_current_rating.copy(),
    'NationalServices': len(quality_current),
    'StateServices': state_current_n.copy()
}
print(f'Current rated services in Australia: {len(quality_current):,}')


In [ ]:
ax = national_current_rating.to_frame('Current').plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color={'Current': 'tab:blue'}
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings in Australia')
plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


#### 6.8.2 Current Ratings: States vs Australia

One figure compares each rating level across states. Orange bars show state percentages. A blue dashed vertical line marks Australia's percentage, labelled above each panel. Each level has its own horizontal scale; n is the state's current rated sample.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)
quality_level_titles = [
    'Significant Improvement\nRequired', 'Working Towards\nNQS',
    'Meeting NQS', 'Exceeding NQS', 'Excellent'
]
for ax, level, title in zip(axes, rating_order, quality_level_titles):
    rating_benchmark = state_current_rating[level].to_frame('State')
    rating_benchmark.plot(
        kind='barh', ax=ax, alpha=0.7, color='tab:orange', legend=False
    )
    national_benchmark = national_current_rating[level]
    ax.axvline(
        national_benchmark, color='tab:blue', linestyle='--',
        linewidth=1.5, label='Australia benchmark', zorder=3
    )
    ax.text(
        1, 1.02, f'Australia: {national_benchmark:.1f}%',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='tab:blue'
    )
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=8)
    ax.set_title(title, fontsize=11, pad=30)
    ax.set_xlabel('Percentage of Services (%)', fontsize=9)
    ax.set_ylabel('')
    ax.set_xlim(0, max(max(rating_benchmark.max().max(), national_benchmark) * 1.3, 1))
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_yticks(range(len(quality_state_order)), [
    f'{state} (n={state_current_n[state]:,})' for state in quality_state_order
])
axes[0].invert_yaxis()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.suptitle('Current ' + quality_topic + ': States vs Australia', fontsize=14)
plt.tight_layout(rect=[0, 0.07, 1, 0.95])
plt.show()


#### 6.8.3 Current Ratings: Service Type Comparison

Compare the percentage of rated services at each level within Centre-Based Care and Family Day Care.


In [ ]:
# Reuse this topic's current recognised ratings, without requiring a previous rating.
service_type_rating_counts = pd.crosstab(
    quality_current[quality_current_column], quality_current['ServiceType']
).reindex(index=rating_order, columns=quality_service_type_order, fill_value=0)
service_type_rated_n = service_type_rating_counts.sum()
service_type_rating_pct = service_type_rating_counts.div(
    service_type_rated_n.replace(0, np.nan), axis='columns'
) * 100

service_type_quality_summary = pd.DataFrame({
    'RatedServices': service_type_rated_n,
    'BelowNQSPct': service_type_rating_pct.loc[
        ['Significant Improvement Required', 'Working Towards NQS']
    ].sum(min_count=1),
    'MeetingOrAbovePct': service_type_rating_pct.loc[
        ['Meeting NQS', 'Exceeding NQS', 'Excellent']
    ].sum(min_count=1),
    'ExceedingOrExcellentPct': service_type_rating_pct.loc[
        ['Exceeding NQS', 'Excellent']
    ].sum(min_count=1)
})
quality_service_type_results['qa7'] = {
    'Counts': service_type_rating_counts.copy(),
    'Percentages': service_type_rating_pct.copy(),
    'Summary': service_type_quality_summary.copy()
}


In [ ]:
ax = service_type_rating_pct.plot(
    kind='barh', figsize=(10, 6), alpha=0.7, color=['tab:blue', 'tab:orange']
)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.xlabel('Percentage of Services (%)')
plt.ylabel(quality_topic + ' Rating')
plt.title('Current ' + quality_topic + ' Ratings by Service Type')
ax.set_xlim(0, max(service_type_rating_pct.max().max() * 1.15, 1))
ax.spines[['top', 'right']].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [
    f'{service_type} (n={service_type_rated_n[service_type]:,})'
    for service_type in quality_service_type_order
], frameon=False)
plt.tight_layout()
plt.show()


#### 6.8.4 Previous vs Current Ratings: Australia

The same paired services are used for both periods.


In [ ]:
australia_qa7_quality = education_sos.dropna(subset=['PreviousQualityArea7Rating','QualityArea7Rating'])
print(f'Number of Australian services: {len(australia_qa7_quality):,}')

In [ ]:
#Previous vs Current QA7 Rating in Australia

# Previous rating distribution
australia_qa7_quality_previous_rating = (
    australia_qa7_quality['PreviousQualityArea7Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Current rating distribution
australia_qa7_quality_current_rating = (
    australia_qa7_quality['QualityArea7Rating']
    .value_counts(normalize=True)
    .reindex(rating_order)
    * 100
)

# Combine
rating_comparison = pd.DataFrame({
    'Previous': australia_qa7_quality_previous_rating,
    'Current': australia_qa7_quality_current_rating
})

# Plot
ax = rating_comparison.plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, width=0.8,
    color={'Previous': 'tab:orange', 'Current': 'tab:blue'}
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=4, fontsize=11
    )

ax.margins(x=0.16)
ax.tick_params(labelsize=11)
plt.xlabel('Percentage of Services (%)', fontsize=12)
plt.ylabel('QA7 Rating')
plt.title('Previous vs Current QA7: Governance and Leadership Ratings in Australia')

plt.gca().spines[['top', 'right']].set_visible(False)
plt.legend(frameon=False)

plt.tight_layout()
plt.show()

### 6.9 National Below-NQS Rate by Quality Area

Compare QA1–QA7 only, excluding Overall Quality. Below NQS means **Working Towards NQS + Significant Improvement Required**; Meeting NQS, Exceeding NQS and Excellent are not counted as below NQS.

For each QA, divide the number below NQS by the number with a recognised current rating for that QA, then multiply by 100. Use the national cleaned sample across all service types. Missing and unassessed ratings are excluded separately for each QA; a previous rating is not required. The labels show the denominator n.


#### 6.9.1 National Below-NQS Rates


In [ ]:
# National current ratings: QA1 to QA7 only, without OverallRating.
qa_below_nqs_ratings = ['Working Towards NQS', 'Significant Improvement Required']
qa_below_nqs_rows = []

for number in range(1, 8):
    ratings = education_sos[f'QualityArea{number}Rating']

    # The denominator includes only recognised ratings for this QA.
    rated_n = ratings.isin(rating_order).sum()
    below_n = ratings.isin(qa_below_nqs_ratings).sum()
    below_rate = below_n / rated_n * 100 if rated_n > 0 else np.nan

    qa_below_nqs_rows.append({
        'QualityArea': f'QA{number}',
        'RatedServices': rated_n,
        'BelowNQSServices': below_n,
        'BelowNQSRate': below_rate
    })

national_qa_below_nqs = pd.DataFrame(qa_below_nqs_rows).set_index('QualityArea')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    national_qa_below_nqs.index,
    national_qa_below_nqs['BelowNQSRate'],
    color='tab:blue', alpha=0.7
)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_yticks(range(len(national_qa_below_nqs)), [
    f'{qa} (n={int(row["RatedServices"]):,})'
    for qa, row in national_qa_below_nqs.iterrows()
])
ax.invert_yaxis()
ax.set_xlim(0, max(national_qa_below_nqs['BelowNQSRate'].max() * 1.2, 1))
ax.set_xlabel('Services Below Meeting NQS (%)')
ax.set_ylabel('Quality Area')
ax.set_title('National Below-NQS Rate by Quality Area')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#### 6.9.2 Interactive Comparison: Region and Service Type

The static chart above is retained. Select a region and ServiceType independently. Each wide pale-orange background bar shows Australia's below-NQS rate for the **same ServiceType**; the narrow blue overlay shows the selected region. A bold black dashed marker identifies the national rate. The fixed comparison column reports selected rate, Australia rate, and their difference for **every QA**. Difference = selected rate minus Australia rate, in percentage points (pp); a positive value means a higher below-NQS rate. For Australia itself, the rates coincide and all differences are zero.

National rates pool service records rather than averaging state rates. Hover for the below-NQS count, rated denominator, national rate and difference. Missing/unassessed ratings are excluded independently for each QA; Overall Quality is not included. This chart uses the current sample, with no previous-rating or end-of-2024 approval requirement. Chart size remains 1000 x 600.


In [ ]:
# Aggregate directly from education_sos; these dictionaries hold chart summaries only.
qa_regions = ['Australia', 'NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']
qa_service_types = ['All service types', 'Centre-Based Care', 'Family Day Care']
qa_names = [f'QA{i}' for i in range(1, 8)]
qa_below_ratings = ['Working Towards NQS', 'Significant Improvement Required']
qa_filter_results = {}

for region in qa_regions:
    qa_filter_results[region] = {}
    for service_type in qa_service_types:
        selected = pd.Series(True, index=education_sos.index)
        if region != 'Australia':
            selected &= education_sos['State'].eq(region)
        if service_type != 'All service types':
            selected &= education_sos['ServiceType'].eq(service_type)
        rates, details = [], []
        for number in range(1, 8):
            column = f'QualityArea{number}Rating'
            rated_n = int((selected & education_sos[column].isin(rating_order)).sum())
            below_n = int((selected & education_sos[column].isin(qa_below_ratings)).sum())
            rates.append(below_n / rated_n * 100 if rated_n else None)
            details.append([region, service_type, below_n, rated_n])
        qa_filter_results[region][service_type] = {'rates': rates, 'details': details}

# Each service type has four traces: national background, selected region,
# a bold national marker, and a fixed summary column for every QA.
# Region updates data; type updates visibility, so the two filters stay independent.
fig_qa_interactive = go.Figure()
qa_positions = list(range(7))
qa_overlay_data = {}
for region in qa_regions:
    qa_overlay_data[region] = {}
    for service_type in qa_service_types:
        national = qa_filter_results['Australia'][service_type]
        selected = qa_filter_results[region][service_type]
        benchmark_x, benchmark_y, benchmark_details = [], [], []
        summaries, selected_details = [], []
        for position, qa in enumerate(qa_names):
            rate = selected['rates'][position]
            national_rate = national['rates'][position]
            difference = rate - national_rate if rate is not None and national_rate is not None else None
            rate_label = f'{rate:.1f}%' if rate is not None else 'No data'
            national_label = f'{national_rate:.1f}%' if national_rate is not None else 'No data'
            difference_label = f'{difference:+.1f} pp' if difference is not None else 'N/A'
            summaries.append(f'<b>{rate_label}</b> | {national_label} | <b>{difference_label}</b>')
            selected_details.append(selected['details'][position] + [qa, national_rate, difference])
            benchmark_x.extend([national_rate, national_rate, None])
            benchmark_y.extend([position - 0.42, position + 0.42, None])
            benchmark_details.extend([national['details'][position] + [qa]] * 3)
        qa_overlay_data[region][service_type] = dict(
            national=national, selected=selected, summaries=summaries,
            details=selected_details, benchmark_x=benchmark_x,
            benchmark_y=benchmark_y, benchmark_details=benchmark_details
        )

for type_number, service_type in enumerate(qa_service_types):
    result = qa_overlay_data['Australia'][service_type]
    visible = type_number == 0
    fig_qa_interactive.add_trace(go.Bar(
        x=result['national']['rates'], y=qa_positions, orientation='h',
        marker_color=plotly_orange, opacity=0.3, width=0.78,
        name='Australia (background)', visible=visible,
        customdata=[detail + [qa] for detail, qa in zip(result['national']['details'], qa_names)],
        hovertemplate=(
            '<b>%{customdata[4]} | Australia</b><br>'
            'Service type: %{customdata[1]}<br>'
            'Below NQS: %{customdata[2]:,} / %{customdata[3]:,}<br>'
            'National rate: %{x:.2f}%<extra></extra>'
        )
    ))
    fig_qa_interactive.add_trace(go.Bar(
        x=result['selected']['rates'], y=qa_positions, orientation='h',
        marker_color=plotly_blue, opacity=0.85, width=0.38,
        name='Selected region', visible=visible, customdata=result['details'],
        hovertemplate=(
            '<b>%{customdata[4]} | %{customdata[0]}</b><br>'
            'Service type: %{customdata[1]}<br>'
            'Below NQS: %{customdata[2]:,} / %{customdata[3]:,}<br>'
            'Selected rate: %{x:.2f}%<br>Australia: %{customdata[5]:.2f}%<br>'
            'Difference: %{customdata[6]:+.2f} percentage points<extra></extra>'
        )
    ))
    fig_qa_interactive.add_trace(go.Scatter(
        x=result['benchmark_x'], y=result['benchmark_y'], mode='lines',
        line=dict(color='black', dash='dash', width=4), connectgaps=False,
        customdata=result['benchmark_details'], name='Australia benchmark', visible=visible,
        hovertemplate=(
            '<b>%{customdata[4]} | Australia benchmark</b><br>'
            'Service type: %{customdata[1]}<br>'
            'Below NQS: %{customdata[2]:,} / %{customdata[3]:,}<br>'
            'National rate: %{x:.2f}%<extra></extra>'
        )
    ))
    fig_qa_interactive.add_trace(go.Scatter(
        x=[0.02] * 7, y=qa_positions, xaxis='x2', mode='text',
        text=result['summaries'], textposition='middle right',
        textfont=dict(size=15, color='black'), cliponaxis=False,
        name='Rate comparison', showlegend=False, visible=visible, hoverinfo='skip'
    ))

qa_state_buttons = []
for region in qa_regions:
    x_values, texts, details = [], [], []
    for service_type in qa_service_types:
        result = qa_overlay_data[region][service_type]
        x_values.extend([result['national']['rates'], result['selected']['rates'],
                         result['benchmark_x'], [0.02] * 7])
        texts.extend([None, None, None, result['summaries']])
        details.extend([
            [detail + [qa] for detail, qa in zip(result['national']['details'], qa_names)],
            result['details'], result['benchmark_details'], None
        ])
    qa_state_buttons.append(dict(label=region, method='update', args=[
        {'x': x_values, 'text': texts, 'customdata': details},
        {'xaxis.autorange': True,
         'title.text': f'Below-NQS Rates: {region} vs Australia'}
    ]))
qa_type_buttons = [dict(label=service_type, method='update', args=[
    {'visible': [trace // 4 == type_number for trace in range(12)]},
    {'xaxis.autorange': True}
]) for type_number, service_type in enumerate(qa_service_types)]

fig_qa_interactive.update_layout(
    template='plotly_white', width=1000, height=600,
    title=dict(text='Below-NQS Rates: Australia vs Australia', y=0.97, x=0.05),
    font=dict(family='Arial', size=16, color='black'),
    barmode='overlay', bargap=0.2,
    xaxis=dict(title='Services below NQS (%)', domain=[0, 0.61], rangemode='tozero',
               gridcolor='whitesmoke', zeroline=False),
    xaxis2=dict(domain=[0.65, 1], range=[0, 1], visible=False, fixedrange=True, anchor='y'),
    yaxis=dict(title='', tickmode='array', tickvals=qa_positions,
               ticktext=qa_names, range=[6.65, -0.7], showgrid=False, zeroline=False),
    margin=dict(l=75, r=20, t=165, b=100),
    legend=dict(orientation='h', x=0, y=-0.19, font=dict(size=14)),
    hoverlabel=dict(bgcolor='white', bordercolor='black', font_color='black', align='left'),
    updatemenus=[
        dict(buttons=qa_state_buttons, x=0.08, y=1.28, xanchor='left', yanchor='top', active=0),
        dict(buttons=qa_type_buttons, x=0.55, y=1.28, xanchor='left', yanchor='top', active=0)
    ],
    annotations=[
        dict(text='Region', x=0, y=1.25, xref='paper', yref='paper', showarrow=False, xanchor='left'),
        dict(text='Type', x=0.47, y=1.25, xref='paper', yref='paper', showarrow=False, xanchor='left'),
        dict(text='<b>Selected | Australia | Difference</b>', x=0.657, y=1.04,
             xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=14)),
        dict(text='Difference in percentage points (pp); positive = higher below-NQS rate.',
             x=0, y=-0.31, xref='paper', yref='paper', showarrow=False,
             xanchor='left', font=dict(size=13, color='dimgray'))
    ]
)
fig_qa_interactive.show()


#### 6.9.3 Below-NQS Rates by Service Type

Compare Centre-Based Care (blue, left) and Family Day Care (orange, right) across QA1-QA7 nationally. Overall Quality is excluded. For each service type and QA separately, divide the number rated Significant Improvement Required or Working Towards NQS by the number with a recognised current rating. Missing and unassessed ratings are excluded for that QA only; a previous rating and an end-of-2024 approval cutoff are not required.

Hover for the percentage, below-NQS count and rated denominator. These are descriptive rates, not adjusted risk estimates. The chart follows the existing 1000 x 600 size and blue/orange palette.


In [ ]:
# National comparison of QA1-QA7; select directly from education_sos.
qa_type_order = ['Centre-Based Care', 'Family Day Care']
qa_type_names = [f'QA{number}' for number in range(1, 8)]
qa_type_below = ['Significant Improvement Required', 'Working Towards NQS']
qa_type_summary = {}

for service_type in qa_type_order:
    type_mask = education_sos['ServiceType'].eq(service_type)
    rates, counts = [], []
    for number in range(1, 8):
        rating = education_sos[f'QualityArea{number}Rating']
        rated_n = int((type_mask & rating.isin(rating_order)).sum())
        below_n = int((type_mask & rating.isin(qa_type_below)).sum())
        rates.append(100 * below_n / rated_n if rated_n else None)
        counts.append([below_n, rated_n])
    qa_type_summary[service_type] = {'rates': rates, 'counts': counts}

fig_qa_service_type = go.Figure()
for service_type, color in zip(qa_type_order, [plotly_blue, plotly_orange]):
    summary = qa_type_summary[service_type]
    fig_qa_service_type.add_trace(go.Bar(
        x=qa_type_names, y=summary['rates'], name=service_type,
        marker_color=color, opacity=0.7,
        text=[f'{rate:.1f}%' if rate is not None else 'No data' for rate in summary['rates']],
        textposition='outside', cliponaxis=False,
        customdata=summary['counts'],
        hovertemplate=(
            '<b>%{x} | ' + service_type + '</b><br>'
            'Below NQS: %{y:.2f}%<br>'
            'Below-NQS services: %{customdata[0]:,}<br>'
            'Rated services: %{customdata[1]:,}<extra></extra>'
        )
    ))

qa_type_max = max(rate for result in qa_type_summary.values()
                  for rate in result['rates'] if rate is not None)
fig_qa_service_type.update_layout(
    template='plotly_white', width=1000, height=600,
    title=dict(text='Below-NQS Rate by Service Type and Quality Area', x=0.05),
    font=dict(family='Arial', size=18, color='black'),
    barmode='group', bargap=0.22, bargroupgap=0.04,
    xaxis=dict(title='Quality Area', categoryorder='array', categoryarray=qa_type_names,
               showgrid=False, tickangle=0),
    yaxis=dict(title='Below NQS (%)', range=[0, qa_type_max * 1.18],
               dtick=5, gridcolor='whitesmoke', zeroline=False),
    legend=dict(title='', orientation='h', x=0.5, xanchor='center', y=-0.22),
    margin=dict(l=85, r=30, t=75, b=110),
    hoverlabel=dict(bgcolor='white', bordercolor='black', font_color='black', align='left')
)
fig_qa_service_type.show()


<a id="section-7"></a>

## 7. Accessibility

Compare three public transport distance measures: train, bus, and the nearest of the two. The following preparation cell calculates the nearest available distance for each service.

Sections 7.1–7.8 follow the same sequence: summary statistics, train-distance distribution, bus-distance distribution, and nearest-distance distribution. Section 7.9 brings the state and national comparisons together.


In [ ]:
education_sos['closest_distance'] = (education_sos[['DistanceToTrainStation_km','DistanceToBusStation_km']]).min(axis=1)

### 7.1 NSW Accessibility


In [ ]:
nsw = education_sos[education_sos['State'] == 'NSW']
overall_distance_summary = (
    nsw.loc[nsw['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    nsw.loc[nsw['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    nsw.loc[nsw['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NSW') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NSW') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NSW Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NSW') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NSW') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NSW Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NSW') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NSW') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NSW Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.2 VIC Accessibility


In [ ]:
vic = education_sos[education_sos['State'] == 'VIC']
overall_distance_summary = (
    vic.loc[vic['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    vic.loc[vic['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    vic.loc[vic['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'VIC') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='VIC') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('VIC Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'VIC') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='VIC') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('VIC Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'VIC') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='VIC') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('VIC Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.3 QLD Accessibility


In [ ]:
qld = education_sos[education_sos['State'] == 'QLD']
overall_distance_summary = (
    qld.loc[qld['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    qld.loc[qld['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    qld.loc[qld['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'QLD') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='QLD') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('QLD Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'QLD') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='QLD') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('QLD Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'QLD') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='QLD') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('QLD Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.4 WA Accessibility


In [ ]:
wa = education_sos[education_sos['State'] == 'WA']
overall_distance_summary = (
    wa.loc[wa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    wa.loc[wa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    wa.loc[wa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'WA') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='WA') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('WA Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'WA') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='WA') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('WA Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'WA') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='WA') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('WA Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.5 SA Accessibility


In [ ]:
sa = education_sos[education_sos['State'] == 'SA']
overall_distance_summary = (
    sa.loc[sa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    sa.loc[sa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    sa.loc[sa['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'SA') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='SA') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('SA Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'SA') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='SA') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('SA Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'SA') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='SA') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('SA Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.6 TAS Accessibility


In [ ]:
tas = education_sos[education_sos['State'] == 'TAS']
overall_distance_summary = (
    tas.loc[tas['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    tas.loc[tas['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    tas.loc[tas['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'TAS') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='TAS') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('TAS Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'TAS') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='TAS') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('TAS Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'TAS') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='TAS') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('TAS Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.7 ACT Accessibility


In [ ]:
act = education_sos[education_sos['State'] == 'ACT']
overall_distance_summary = (
    act.loc[act['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    act.loc[act['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    act.loc[act['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'ACT') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='ACT') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('ACT Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'ACT') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='ACT') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('ACT Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'ACT') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='ACT') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('ACT Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.8 NT Accessibility


In [ ]:
nt = education_sos[education_sos['State'] == 'NT']
overall_distance_summary = (
    nt.loc[nt['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    nt.loc[nt['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    nt.loc[nt['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NT') & (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NT') & (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NT Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NT') & (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NT') & (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NT Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['State'] == 'NT') & (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['State']=='NT') & (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('NT Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

### 7.9 Australia and State Summary

Compare Urban and Rural distances across states and nationally. Outliers are hidden from the boxplots but remain in the data.


#### 7.9.1 Distance Distributions: State and National Boxplots


In [ ]:
state_order = ['NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']
access_metrics = {
    'closest_distance': 'Nearest Train or Bus Reference',
    'DistanceToTrainStation_km': 'Train Reference',
    'DistanceToBusStation_km': 'Bus Reference'
}

# One national summary figure covering all three distance measures.
fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharey=True)
comparison_order = state_order + ['Australia']

for ax, (column, label) in zip(axes, access_metrics.items()):
    urban_values, rural_values = [], []
    urban_positions, rural_positions = [], []
    urban_n, rural_n = {}, {}

    for position, state in enumerate(comparison_order):
        state_mask = pd.Series(True, index=education_sos.index) if state == 'Australia' else education_sos['State'].eq(state)
        urban = education_sos.loc[
            state_mask & education_sos['Classification'].eq('Urban'), column
        ].dropna()
        rural = education_sos.loc[
            state_mask & education_sos['Classification'].eq('Rural'), column
        ].dropna()
        urban_n[state], rural_n[state] = len(urban), len(rural)
        if not urban.empty:
            urban_values.append(urban)
            urban_positions.append(position + 0.19)
        if not rural.empty:
            rural_values.append(rural)
            rural_positions.append(position - 0.19)

    urban_plot = ax.boxplot(
        urban_values, positions=urban_positions, vert=False, widths=0.32,
        showfliers=False, patch_artist=True, manage_ticks=False
    )
    rural_plot = ax.boxplot(
        rural_values, positions=rural_positions, vert=False, widths=0.32,
        showfliers=False, patch_artist=True, manage_ticks=False
    )
    for box in urban_plot['boxes']:
        box.set_facecolor(location_colors['Urban'])
        box.set_alpha(0.7)
    for box in rural_plot['boxes']:
        box.set_facecolor(location_colors['Rural'])
        box.set_alpha(0.7)

    ax.set_title(label)
    ax.set_xlabel('Distance (km)')
    ax.set_yticks(range(len(comparison_order)))
    ax.set_yticklabels([
        f'{state}\nU n={urban_n[state]:,}; R n={rural_n[state]:,}'
        for state in comparison_order
    ])
    ax.set_ylim(len(comparison_order) - 0.5, -0.5)
    ax.grid(False)
    ax.spines[['top', 'right']].set_visible(False)

axes[-1].legend(
    [summary_box for summary_box in [rural_plot['boxes'][0], urban_plot['boxes'][0]]],
    location_order, loc='lower right'
)
fig.suptitle('Public Transport Reference Distance: State and Australia Summary')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


#### 7.9.2 National Summary Statistics and Distributions


In [ ]:
overall_distance_summary = (
    education_sos.loc[education_sos['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['closest_distance'])
    .groupby('Classification')['closest_distance']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
train_distance_summary = (
    education_sos.loc[education_sos['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToTrainStation_km'])
    .groupby('Classification')['DistanceToTrainStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
Bus_distance_summary = (
    education_sos.loc[education_sos['Classification'].isin(['Urban', 'Rural'])]
    .dropna(subset=['DistanceToBusStation_km'])
    .groupby('Classification')['DistanceToBusStation_km']
    .agg(
        Mean='mean',
        Min='min',
        Q1=lambda x: x.quantile(0.25),
        Median='median',
        Q3=lambda x: x.quantile(0.75),
        Max='max'
    )
    .round(2)
)
print(overall_distance_summary)
print(train_distance_summary)
print(Bus_distance_summary)

# Save the pooled national median once; Section 7.10 reuses this full-precision value.
# The Urban/Rural medians above are separate groups, not the national median.
national_median_nearest_distance = education_sos.loc[
    np.isfinite(education_sos['closest_distance']) & education_sos['closest_distance'].ge(0),
    'closest_distance'
].median()
print(f'National median nearest distance: {national_median_nearest_distance:.2f} km')


In [ ]:
urban = education_sos.loc[
    (education_sos['Classification']=='Urban'),
    'DistanceToTrainStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['Classification']=='Rural'),
    'DistanceToTrainStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('Australia Train Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()


In [ ]:
urban = education_sos.loc[
    (education_sos['Classification']=='Urban'),
    'DistanceToBusStation_km'
].dropna()
rural = education_sos.loc[
    (education_sos['Classification']=='Rural'),
    'DistanceToBusStation_km'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('Australia Bus Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
urban = education_sos.loc[
    (education_sos['Classification']=='Urban'),
    'closest_distance'
].dropna()
rural = education_sos.loc[
    (education_sos['Classification']=='Rural'),
    'closest_distance'
].dropna()
# the distribution can be better show with the log transfromation and since all value is bigger than 0 given distance, its a possible choice. to calculate better at the small value less than 1, use log(1+value)
# this part decided to use a percentage vertical line as the sample size is quit different between urban and rural area
# pd.concat use to unify the x-axis for urban and rural list so that it can be grouped and seperated from the same 
bins = np.histogram_bin_edges(np.log(1+ pd.concat([urban, rural])), bins=80)
fig, ax = plt.subplots(figsize=(10, 6))
for label, values in [('Rural', rural), ('Urban', urban)]:
    heights, edges, bars = ax.hist(
        np.log(1+values), bins=bins, weights=np.full(len(values), 100 / len(values)),
        alpha=0.7, label=label, color=location_colors[label]
    )
plt.title('Australia Overall Accessibility: Urban vs Rural')
plt.xlabel('ln(1 + distance in km)')
plt.ylabel('Services within each group (%)')
plt.xlim(left=0)
plt.tight_layout()
plt.legend()
plt.show()

#### 7.9.3 Distance and Rural-to-Urban Gap: Interactive Comparison


In [ ]:
# Use medians because the distance distribution is skewed and extreme values affect means.
state_access_distance = (education_sos.loc[
        education_sos['Classification'].isin(['Urban', 'Rural'])
    ]
    .dropna(subset=['State', 'closest_distance'])
    .groupby(['State', 'Classification'])['closest_distance']
    .median()
    .unstack()
)

state_access_distance['Urban_Rural_Gap'] = (
    state_access_distance['Rural'] / state_access_distance['Urban']
)

state_median_distance = (
    education_sos
    .dropna(subset=['State', 'closest_distance'])
    .groupby('State')['closest_distance']
    .median()
)

state_access_distance['Median_Distance'] = state_median_distance
state_access_distance = state_access_distance.dropna(
    subset=['Urban_Rural_Gap', 'Median_Distance']
)
national_median_distance = (
    education_sos.groupby('Classification')['closest_distance']
    .median()
)
urban_median_dis = national_median_distance['Urban']
rural_median_dis = national_median_distance['Rural']

x_benchmark = rural_median_dis / urban_median_dis
y_benchmark = national_median_nearest_distance

# Interactive scatter: reuse the same state summary and national medians.
fig_access_scatter = go.Figure(go.Scatter(
    x=state_access_distance['Urban_Rural_Gap'].tolist(),
    y=state_access_distance['Median_Distance'].tolist(),
    mode='markers', text=state_access_distance.index.tolist(),
    marker=dict(size=11, color=plotly_blue, opacity=0.7),
    customdata=state_access_distance[['Urban', 'Rural']].to_numpy().tolist(),
    hovertemplate=(
        '<b>%{text}</b><br>Rural/Urban median distance: %{x:.2f}x<br>'
        'State median distance: %{y:.2f} km<br>'
        'Urban median: %{customdata[0]:.2f} km<br>'
        'Rural median: %{customdata[1]:.2f} km<extra></extra>'
    )
))
fig_access_scatter.add_vline(x=x_benchmark, line_dash='dash', line_color='gray')
fig_access_scatter.add_hline(y=y_benchmark, line_dash='dash', line_color='gray')
fig_access_scatter.add_annotation(
    x=x_benchmark, y=y_benchmark,
    text=f'National median benchmark<br>({x_benchmark:.2f}x, {y_benchmark:.2f} km)',
    showarrow=False, xanchor='left', yanchor='bottom', xshift=8, yshift=6,
    bgcolor='white', font=dict(size=11)
)
fig_access_scatter.update_layout(
    template='plotly_white', width=1000, height=600,
    title='State Accessibility: Distance and Rural-to-Urban Gap',
    xaxis_title='Rural-to-Urban Median Distance Ratio (x)',
    yaxis_title='Median Distance to Nearest Public Transport (km)',
    margin=dict(l=100, r=90, t=80, b=130), showlegend=False
)
fig_access_scatter.update_xaxes(showgrid=False, zeroline=False)
fig_access_scatter.update_yaxes(showgrid=False, zeroline=False)
fig_access_scatter.add_annotation(
    x=0.5, y=-0.23, xref='paper', yref='paper', showarrow=False,
    text='Medians reduce the influence of extreme transport distances.', font=dict(size=11)
)
# Place every state label below its point; white backgrounds keep benchmark lines clear.
for state, row in state_access_distance.iterrows():
    fig_access_scatter.add_annotation(
        x=row['Urban_Rural_Gap'], y=row['Median_Distance'], text=state,
        showarrow=False, xanchor='center', yanchor='top', yshift=-10,
        bgcolor='white', borderpad=1, font=dict(size=12)
    )

fig_access_scatter.show()


### 7.10 Public Transport Coverage within the National Median Nearest Distance

Compare all eight states and territories using `education_sos`. For each service, take the smaller valid distance from `DistanceToBusStation_km` and `DistanceToTrainStation_km`. There is no quality-rating filter, urban-only filter or end-of-2024 cutoff.

- **Shared threshold:** `national_median_nearest_distance` is saved in the preceding Australia summary cell and reused here. It is the median across all valid service-level `closest_distance` values, not an average of state or Urban/Rural medians. Calculations use full precision; only labels are rounded.
- **Coverage:** services whose nearest recorded bus or train station is **<= the national median nearest distance**, divided by services with at least one valid distance in the same state. Missing, negative and infinite distances are excluded, not counted as uncovered. Hover shows numerator, denominator and missing counts.
- **Nearest distance:** the median of each service's nearest recorded bus/train distance in that state, in km; hover includes Q1 and Q3. This is not the minimum distance among all services in the state.
- **National reference:** the dashed line uses the pooled Australian service records. It is not the average of eight state percentages or medians.

**Scope:** this measures proximity to the recorded bus/train references, not verified route availability or travel time. The supplied station list is not established as a complete inventory of public transport stops; it includes bus depot/terminal labels and does not provide separate ferry/tram distances. Do not interpret the result as comprehensive public-transport reachability. If one mode is missing, use the available distance and interpret coverage cautiously.

Existing analysis cells are retained. Both new figures use the current blue/orange palette and 1000 x 600 layout.

The national median is less sensitive to extreme distances than the mean. This is a descriptive, data-derived comparison threshold, not a policy standard or a measure of actual transit journey accessibility.


In [ ]:
# Reuse education_sos; no quality, rail-only, urban-only or approval-date filter.
transport_state_order = ['NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']

# Reuse the nearest-distance field and national median from earlier accessibility cells.
# Invalid/missing distances remain excluded; the threshold is not rounded.
transport_nearest_km = pd.to_numeric(education_sos['closest_distance'], errors='coerce')
transport_nearest_km = transport_nearest_km.where(
    np.isfinite(transport_nearest_km) & transport_nearest_km.ge(0)
)
transport_summary_rows = []
for region in transport_state_order + ['Australia']:
    region_mask = education_sos['State'].eq(region) if region != 'Australia' else education_sos['State'].isin(transport_state_order)
    distances = transport_nearest_km.loc[region_mask].dropna()
    total_n = int(region_mask.sum())
    valid_n = len(distances)
    within_n = int(distances.le(national_median_nearest_distance).sum())  # Same national threshold for all states.
    transport_summary_rows.append({
        'State': region, 'TotalServices': total_n, 'ValidDistanceServices': valid_n,
        'MissingDistanceServices': total_n - valid_n, 'WithinNationalMedianServices': within_n,
        'WithinNationalMedianPercent': 100 * within_n / valid_n if valid_n else np.nan,
        'MedianNearestKm': distances.median(), 'Q1NearestKm': distances.quantile(0.25),
        'Q3NearestKm': distances.quantile(0.75)
    })
transport_state_summary = pd.DataFrame(transport_summary_rows).set_index('State')


In [ ]:
# Two separate 1000 x 600 views: coverage percentage and typical nearest distance.
# Australia is calculated from all service records, not an unweighted state average.
transport_figures = {}
# Build hover labels with ordinary Python strings and named table columns.
# <br> adds a line break; Plotly inserts the finished string via %{hovertext}.
transport_hover_text = []
for state in transport_state_order:
    row = transport_state_summary.loc[state]
    transport_hover_text.append('<br>'.join([
        state,
        f"National median threshold: {national_median_nearest_distance:.2f} km",
        f"Within national median: {row['WithinNationalMedianPercent']:.2f}%",
        f"Services within threshold: {row['WithinNationalMedianServices']:,.0f}",
        f"Services with distance: {row['ValidDistanceServices']:,.0f}",
        f"All services: {row['TotalServices']:,.0f}",
        f"Missing distance: {row['MissingDistanceServices']:,.0f}",
        f"Median nearest distance: {row['MedianNearestKm']:.2f} km",
        f"Q1 / Q3: {row['Q1NearestKm']:.2f} / {row['Q3NearestKm']:.2f} km"
    ]))
transport_chart_specs = [
    ('WithinNationalMedianPercent', 'public_transport_within_national_median',
     f'Services within National Median Nearest Distance ({national_median_nearest_distance:.2f} km)',
     f'Services within {national_median_nearest_distance:.2f} km (%)',
     plotly_blue, plotly_orange, '.1f', '%'),
    ('MedianNearestKm', 'public_transport_nearest_distance',
     'Nearest Recorded Public Transport Distance by State', 'Median nearest distance (km)',
     plotly_orange, plotly_blue, '.2f', ' km')
]
for metric, chart_key, title, axis_title, color, benchmark_color, number_format, unit in transport_chart_specs:
    values = transport_state_summary.loc[transport_state_order, metric]
    national_value = float(transport_state_summary.loc['Australia', metric])
    fig_transport = go.Figure(go.Bar(
        x=transport_state_order, y=values.tolist(), marker_color=color, opacity=0.7,
        text=[format(value, number_format) + unit for value in values],
        textposition='none',
        insidetextanchor='end', insidetextfont=dict(color='black'), cliponaxis=False,
        hovertext=transport_hover_text,
        hovertemplate='%{hovertext}<extra></extra>'
    ))
    fig_transport.add_hline(y=national_value, line_dash='dash', line_color=benchmark_color, line_width=2)
    fig_transport.add_annotation(
        x=1, y=1.06, xref='paper', yref='paper', showarrow=False,
        xanchor='right', text='Australia: ' + format(national_value, number_format) + unit + ' (dashed line)',
        font=dict(color=benchmark_color, size=18)
    )
    fig_transport.update_layout(
        template='plotly_white', width=1000, height=600,
        title=dict(text=title, x=0.05), font=dict(family='Arial', size=18, color='black'),
        xaxis=dict(title='State / Territory', categoryorder='array', categoryarray=transport_state_order,
                   tickangle=0, showgrid=False),
        yaxis=dict(title=axis_title, range=[0, 100 if metric == 'WithinNationalMedianPercent' else max(values.max(), national_value) * 1.25],
                   gridcolor='whitesmoke', zeroline=False),
        bargap=0.35, showlegend=False, margin=dict(l=100, r=35, t=110, b=80),
        hoverlabel=dict(bgcolor='white', bordercolor='black', font_color='black', align='left')
    )
    # White-backed value labels remain legible around the national reference line.
    for state, value in values.items():
        fig_transport.add_annotation(
            x=state, y=float(value), text=format(value, number_format) + unit,
            showarrow=False, yanchor='bottom', yshift=5,
            bgcolor='white', borderpad=2, font=dict(size=18, color='black')
        )
    transport_figures[chart_key] = fig_transport
    fig_transport.show()


<a id="section-8"></a>

## 8. Estimated Demand and Accessibility



### 8.1 Prepare Estimated Children and Service Summary

The sum of 2011-2024 birth registrations approximates children aged 0-13 at the end of 2024; it is not a resident-population estimate and does not adjust for migration or deaths. All service types are included, restricted to approvals on or before 31 December 2024. Children per service is total estimated children divided by the count of services in that state.

The preparation table retains its original capacity summary. The following scatter instead uses observed `closest_distance`, the nearest recorded bus/train distance, excluding missing, negative or infinite distances from the vertical-axis calculation. Its horizontal denominator still includes all services in the cohort. Hover shows both counts.

State distances and the national distance reference use the median by default. The national reference pools service-level distances within the same 2024 cohort; it is not an average of state medians. The horizontal national reference uses the workbook's Australia total and the national service count, including ACT. Recorded proximity does not establish route availability or complete public-transport coverage.


In [ ]:
# Part 1: Calculate children and service capacity by state

# Birth cohorts: approximate children aged 0–13 at the end of 2024
births_2011_2024 = pd.read_excel('ABS Birth estimation.xlsx')
births_2011_2024.columns = (births_2011_2024.columns
    .astype(str)
    .str.strip()
)
births_2011_2024['states'] = (births_2011_2024['states']
    .str.strip()
)

birth_years = [str(year) for year in range(2011, 2025)]

births_2011_2024[birth_years] = (births_2011_2024[birth_years]
    .apply(pd.to_numeric, errors='raise')
)

births_2011_2024['EstimatedChildren_0_13'] = (births_2011_2024[birth_years]
    .sum(axis=1)
)
births_2011_2024['State'] = (births_2011_2024['states']
    .map(state_names)
)

children_by_state = (births_2011_2024
    .dropna(subset=['State'])
    .set_index('State')['EstimatedChildren_0_13']
)
national_children = births_2011_2024.loc[births_2011_2024['states'] =='Australia',
    'EstimatedChildren_0_13'
].item()
services_2024 = education_sos.loc[education_sos['ApprovalDate'] <= '2024-12-31']

# State-level service capacity
children_capacity_2024 = (
    services_2024
    .groupby('State')
    .agg(
        Services=('ServiceApprovalNumber', 'nunique'),
        MedianApprovedPlaces=('ApprovedPlaces_Clean', 'median')
    )
    .join(children_by_state)
)

# Children per service
children_capacity_2024['ChildrenPerService'] = (
    children_capacity_2024['EstimatedChildren_0_13']
    / children_capacity_2024['Services']
)

# National benchmarks
national_service_count = (
    services_2024['ServiceApprovalNumber']
    .nunique()
)

national_children_per_service = (
    national_children / national_service_count
)

national_median_places = (
    services_2024['ApprovedPlaces_Clean']
    .median()
)

# Final table
children_capacity_2024 = children_capacity_2024[
    [
        'EstimatedChildren_0_13',
        'Services',
        'ChildrenPerService',
        'MedianApprovedPlaces'
    ]
]

display(children_capacity_2024.round(2))

### 8.2 Estimated Children per Service and Median Transport Distance

Use `closest_distance` for the vertical axis. The shared 2024 approval cohort is retained. Change `distance_statistic` from `median` to `mean` to switch the state summaries and national distance reference together.


In [ ]:
# Part 2: Estimated children per service vs nearest public transport distance.
# Use 'median' for the middle distance, or change this one value to 'mean' for average.
distance_statistic = 'median'
distance_label = 'Median' if distance_statistic == 'median' else 'Mean'

# Reuse the same services approved by the end of 2024 as the horizontal axis.
# This benchmark therefore uses the 2024 cohort, rather than the full current sample.
distance_2024 = pd.to_numeric(services_2024['closest_distance'], errors='coerce')
distance_2024 = distance_2024.where(np.isfinite(distance_2024) & distance_2024.ge(0))
state_distance_2024 = (
    distance_2024.groupby(services_2024['State'])
    .agg(['count', distance_statistic])
    .rename(columns={'count': 'DistanceServices', distance_statistic: 'ClosestDistanceKm'})
)
plot_data = children_capacity_2024.join(state_distance_2024).dropna(
    subset=['ChildrenPerService', 'ClosestDistanceKm']
)
national_transport_distance_2024 = distance_2024.agg(distance_statistic)

# Build hover text with named columns. <br> is a line break in the tooltip.
children_access_hover = []
for state, row in plot_data.iterrows():
    children_access_hover.append('<br>'.join([
        state,
        f"Estimated children per service: {row['ChildrenPerService']:.1f}",
        f"{distance_label} nearest transport distance: {row['ClosestDistanceKm']:.2f} km",
        f"All services in the 2024 cohort: {row['Services']:,.0f}",
        f"Services with a valid distance: {row['DistanceServices']:,.0f}",
        f"Estimated children (birth-cohort proxy): {row['EstimatedChildren_0_13']:,.0f}"
    ]))

fig_children_accessibility = go.Figure(go.Scatter(
    x=plot_data['ChildrenPerService'].tolist(),
    y=plot_data['ClosestDistanceKm'].tolist(),
    mode='markers+text', text=plot_data.index.tolist(),
    textposition=['bottom right' if state == 'NSW' else
                  'top left' if state in ['TAS', 'WA'] else 'top right'
                  for state in plot_data.index],
    textfont=dict(size=16), cliponaxis=False,
    marker=dict(size=12, color=plotly_blue, opacity=0.7),
    hovertext=children_access_hover,
    hovertemplate='%{hovertext}<extra></extra>'
))
fig_children_accessibility.add_vline(
    x=national_children_per_service, line_dash='dash', line_color='gray'
)
fig_children_accessibility.add_hline(
    y=national_transport_distance_2024, line_dash='dash', line_color='gray'
)
fig_children_accessibility.add_annotation(
    x=0.98, y=1.08, xref='paper', yref='paper',
    text=(f'Australia: {national_children_per_service:.1f} children/service | '
          f'{distance_label.lower()} distance {national_transport_distance_2024:.2f} km'),
    showarrow=False, xanchor='right', yanchor='bottom',
    bgcolor='white', font=dict(size=15)
)
fig_children_accessibility.update_layout(
    template='plotly_white', width=1000, height=600,
    title=dict(text='Estimated Children per Service and Public Transport Distance',
               x=0.05, font=dict(size=21)),
    font=dict(family='Arial', size=16, color='black'),
    xaxis_title='Estimated Children per Service',
    yaxis_title=f'{distance_label} Nearest Public Transport Distance (km)',
    margin=dict(l=100, r=60, t=130, b=120), showlegend=False,
    hoverlabel=dict(bgcolor='white', bordercolor='black', font_color='black', align='left')
)
fig_children_accessibility.update_xaxes(
    showgrid=False, zeroline=False,
    range=[plot_data['ChildrenPerService'].min() * 0.92,
           plot_data['ChildrenPerService'].max() * 1.09]
)
fig_children_accessibility.update_yaxes(
    showgrid=False, zeroline=False,
    range=[0, plot_data['ClosestDistanceKm'].max() * 1.20]
)
fig_children_accessibility.add_annotation(
    x=0, y=-0.27, xref='paper', yref='paper', showarrow=False, xanchor='left',
    text='Services approved by end of 2024; nearest recorded bus/train distance, excluding missing values.<br>'
         'Children per service uses a birth-cohort proxy, not a current resident-population estimate.',
    font=dict(size=12, color='dimgray')
)
fig_children_accessibility.show()


<a id="section-9"></a>

## 9. Operational Trends

**Core question:** What activities do services provide, and how does capacity vary with activity type and activity count?

| Subsection | Focus |
|---|---|
| 9.1 Service Activity Profile | State cases, Australia, comparisons and exact activity combinations |
| 9.2 Capacity by Service Activity | Interactive boxplot and descriptive summaries |
| 9.3 Capacity by Number of Activities | Interactive boxplot and descriptive summaries |

State and national activity profiles use `operational_data`. Supplementary activity combinations use the end-of-2024 `operations` sample. Both capacity boxplots select the end-of-2024 CBC sample directly from `education_sos` using the shared `capacity_analysis_mask`.

Capacities are current recorded attributes of the selected approval cohort, not reconstructed historical observations. Approval-date trends are covered in Section 5.


### Prepare Operational Fields

Select all consecutive columns from `Long Day Care` through `Holiday Care Sunday End Time` (inclusive), and add `ServiceApprovalNumber`, `State`, and `Classification`. Use `education_raw` to retain the full column range, including schedule fields removed during earlier missingness filtering; match the rows to `education_sos` by service ID.


In [ ]:
# Select every column from Long Day Care to Holiday Care Sunday End Time.
operational_columns = education_raw.loc[
    :, 'Long Day Care':'Holiday Care Sunday End Time'
].columns.tolist()

# Add the service ID, state and Urban/Rural classification.
operational_data = education_sos[
    ['ServiceApprovalNumber', 'State', 'Classification']
].merge(
    education_raw[['ServiceApprovalNumber'] + operational_columns],
    on='ServiceApprovalNumber',
    how='left',
    validate='one_to_one'
)


### 9.1 Service Activity Profile

**What activities do approved services provide?** Each Yes/No activity flag is considered separately. A service may provide several activities, so activity percentages can sum to more than 100%. State and Australia cases use all corresponding services in `operational_data`; Urban/Rural percentages use each location group as the denominator. Exact activity combinations provide a second, mutually exclusive description; they do not replace the individual flags.


In [ ]:
# NSW 案例：
# Service activity columns
activity_cols = [
    'Long Day Care',
    'Preschool/Kindergarten - Part of a School',
    'Preschool/Kindergarten - Stand alone',
    'Outside school Hours Care - After School',
    'Outside school Hours Care - Before School',
    'Outside school Hours Care - Vacation Care',
    'Other'
]

operational_data[activity_cols] = (
    operational_data[activity_cols]
    .replace({'Yes': 1, 'No': 0})
)
pd.unique(operational_data[activity_cols].values.ravel())


#### 9.1.1 NSW


In [ ]:
nsw = operational_data.loc[operational_data['State'] == 'NSW']
nsw_activity_count = (
    nsw[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(nsw.shape[0])
nsw_activity_count

In [ ]:
nsw = operational_data.loc[
    operational_data['State'] == 'NSW'
]

nsw['ActivityCount'] = nsw[activity_cols].sum(axis=1)

nsw['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    nsw_activity_count.index,
    nsw_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in NSW')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
nsw_activity_location = (
    nsw
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
nsw_activity_location = nsw_activity_location.T
nsw_activity_location

In [ ]:
ax = nsw_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural NSW')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    nsw['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in NSW')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        nsw['ActivityCount'],
        nsw['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural NSW')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.2 VIC


In [ ]:
vic = operational_data.loc[operational_data['State'] == 'VIC']
vic_activity_count = (
    vic[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(vic.shape[0])
vic_activity_count

In [ ]:
vic = operational_data.loc[
    operational_data['State'] == 'VIC'
]

vic['ActivityCount'] = vic[activity_cols].sum(axis=1)

vic['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    vic_activity_count.index,
    vic_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in VIC')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
vic_activity_location = (
    vic
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
vic_activity_location = vic_activity_location.T
vic_activity_location

In [ ]:
ax = vic_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural VIC')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    vic['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in VIC')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        vic['ActivityCount'],
        vic['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural VIC')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.3 QLD


In [ ]:
qld = operational_data.loc[operational_data['State'] == 'QLD']
qld_activity_count = (
    qld[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(qld.shape[0])
qld_activity_count

In [ ]:
qld = operational_data.loc[
    operational_data['State'] == 'QLD'
]

qld['ActivityCount'] = qld[activity_cols].sum(axis=1)

qld['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    qld_activity_count.index,
    qld_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in QLD')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
qld_activity_location = (
    qld
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
qld_activity_location = qld_activity_location.T
qld_activity_location

In [ ]:
ax = qld_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity: Urban vs Rural QLD')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    qld['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in QLD')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        qld['ActivityCount'],
        qld['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural QLD')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.4 WA


In [ ]:
wa = operational_data.loc[operational_data['State'] == 'WA']
wa_activity_count = (
    wa[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(wa.shape[0])
wa_activity_count

In [ ]:
wa = operational_data.loc[
    operational_data['State'] == 'WA'
]

wa['ActivityCount'] = wa[activity_cols].sum(axis=1)

wa['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    wa_activity_count.index,
    wa_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in WA')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
wa_activity_location = (
    wa
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
wa_activity_location = wa_activity_location.T
wa_activity_location

In [ ]:
ax = wa_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural WA')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    wa['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in WA')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        wa['ActivityCount'],
        wa['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural WA')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.5 SA


In [ ]:
sa = operational_data.loc[operational_data['State'] == 'SA']
sa_activity_count = (
    sa[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(sa.shape[0])
sa_activity_count

In [ ]:
sa = operational_data.loc[
    operational_data['State'] == 'SA'
]

sa['ActivityCount'] = sa[activity_cols].sum(axis=1)

sa['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    sa_activity_count.index,
    sa_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in SA')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
sa_activity_location = (
    sa
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
sa_activity_location = sa_activity_location.T
sa_activity_location

In [ ]:
ax = sa_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural SA')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    sa['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in SA')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        sa['ActivityCount'],
        sa['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural SA')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.6 TAS


In [ ]:
tas = operational_data.loc[operational_data['State'] == 'TAS']
tas_activity_count = (
    tas[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(tas.shape[0])
tas_activity_count

In [ ]:
tas = operational_data.loc[
    operational_data['State'] == 'TAS'
]

tas['ActivityCount'] = tas[activity_cols].sum(axis=1)

tas['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    tas_activity_count.index,
    tas_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in TAS')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
tas_activity_location = (
    tas
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
tas_activity_location = tas_activity_location.T
tas_activity_location

In [ ]:
ax = tas_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural TAS')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    tas['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in TAS')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        tas['ActivityCount'],
        tas['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural TAS')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.7 ACT


In [ ]:
act = operational_data.loc[operational_data['State'] == 'ACT']
act_activity_count = (
    act[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(act.shape[0])
act_activity_count

In [ ]:
act = operational_data.loc[
    operational_data['State'] == 'ACT'
]

act['ActivityCount'] = act[activity_cols].sum(axis=1)

act['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    act_activity_count.index,
    act_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in ACT')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
act_activity_location = (
    act
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
act_activity_location = act_activity_location.T
act_activity_location

In [ ]:
ax = act_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural ACT')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    act['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in ACT')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        act['ActivityCount'],
        act['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural ACT')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.8 NT


In [ ]:
nt = operational_data.loc[operational_data['State'] == 'NT']
nt_activity_count = (
    nt[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(nt.shape[0])
nt_activity_count

In [ ]:
nt = operational_data.loc[
    operational_data['State'] == 'NT'
]

nt['ActivityCount'] = nt[activity_cols].sum(axis=1)

nt['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    nt_activity_count.index,
    nt_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in NT')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
nt_activity_location = (
    nt
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
nt_activity_location = nt_activity_location.T
nt_activity_location

In [ ]:
ax = nt_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural NT')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    nt['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in NT')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        nt['ActivityCount'],
        nt['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural NT')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.9 Australia

The same case analysis uses all services in `operational_data`; only the state filter is removed. Percentages within each Urban/Rural group use that group's service count. Activities may overlap.


In [ ]:
australia = operational_data.copy()
australia_activity_count = (
    australia[activity_cols]
    .sum()
    .sort_values(ascending=False)
)

print(australia.shape[0])
australia_activity_count

In [ ]:
australia = operational_data.copy()

australia['ActivityCount'] = australia[activity_cols].sum(axis=1)

australia['ActivityCount'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    australia_activity_count.index,
    australia_activity_count.values,
    alpha=0.7,
    height = 0.5
)

plt.bar_label(
    bars,
)

plt.xlabel('Number of Services')
plt.ylabel('Service Activity')
plt.title('Service Activities Provided in Australia')

plt.gca().invert_yaxis()

plt.show()

In [ ]:
australia_activity_location = (
    australia
    .groupby('Classification')[activity_cols]
    .mean()
    * 100
)
australia_activity_location = australia_activity_location.T
australia_activity_location

In [ ]:
ax = australia_activity_location.reindex(columns=location_order).plot(
    kind='barh',
    figsize=(10, 6),
    alpha=0.7, color=location_colors
)

plt.xlabel('Percentage of Services (%)')
plt.ylabel('Service Activity')
plt.title('Service Activity Mix: Urban vs Rural Australia')

plt.gca().invert_yaxis()

# Label both Rural and Urban bars without changing their order or colours.
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.margins(x=0.16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

ax = (
    australia['ActivityCount']
    .value_counts()
    .sort_index()
    .plot(
        kind='bar',
        alpha=0.7
    )
)

ax.bar_label(
    ax.containers[0]
)

plt.xlabel('Number of Activities Provided')
plt.ylabel('Number of Services')
plt.title('Number of Activities Provided per Service in Australia')
plt.xticks(rotation=0)

plt.show()

In [ ]:
activity_count_location = (
    pd.crosstab(
        australia['ActivityCount'],
        australia['Classification'],
        normalize='columns'
    )
    * 100
)

activity_count_location

In [ ]:
ax = activity_count_location.reindex(columns=location_order).plot(
    kind='bar',
    figsize=(10, 6),
    width=0.8,
    alpha=0.7, color=location_colors
)

for group_number, container in enumerate(ax.containers):
    ax.bar_label(
        container,
        fmt='%.1f%%',
        fontsize=11, padding=4 if group_number == 0 else 19
    )

plt.xlabel('Number of Activities Provided')
plt.ylabel('Percentage of Services (%)')
plt.title('Number of Activities per Service: Urban vs Rural Australia')
plt.xticks(rotation=0)

# Stagger neighbouring value labels while retaining Rural left / Urban right.
ax.margins(y=0.18)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

#### 9.1.10 State Comparison: Activity Mix and Breadth

This single visual summarises the same state cases using percentages, so larger states do not dominate. The first panel shows the share of services providing each activity. The second shows the share providing two or more activities. The third shows each activity's Rural minus Urban difference in percentage points (positive = more common in Rural services). These are descriptive patterns, not quality rankings. All panels use `operational_data`; Australia is computed from all service records, not the average of state percentages. Activity groups overlap, and percentages across activities need not sum to 100%.


In [ ]:
# Aggregate the state cases using the same sample and activity flags.
activity_state_order = ['NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']
activity_short_labels = ['Long Day Care', 'Preschool - School', 'Preschool - Standalone',
                         'OSHC - After School', 'OSHC - Before School', 'OSHC - Vacation', 'Other']
state_activity_percent = operational_data.groupby('State')[activity_cols].mean().reindex(activity_state_order) * 100
state_activity_percent.loc['Australia'] = operational_data[activity_cols].mean() * 100
activity_state_sizes = operational_data.groupby('State').size().reindex(activity_state_order)
activity_state_sizes.loc['Australia'] = len(operational_data)

activity_case_summary = operational_data[['State', 'Classification']].copy()
activity_case_summary['ActivityCount'] = operational_data[activity_cols].sum(axis=1)
activity_case_summary['MultipleActivities'] = activity_case_summary['ActivityCount'].ge(2)
state_multiple_activity_pct = activity_case_summary.groupby('State')['MultipleActivities'].mean().reindex(activity_state_order) * 100
state_multiple_activity_pct.loc['Australia'] = activity_case_summary['MultipleActivities'].mean() * 100

state_activity_by_location = operational_data.loc[
    operational_data['Classification'].isin(['Urban', 'Rural'])
].groupby(['State', 'Classification'])[activity_cols].mean() * 100
state_activity_location_gap = (
    state_activity_by_location.xs('Rural', level='Classification')
    - state_activity_by_location.xs('Urban', level='Classification')
).reindex(activity_state_order)
location_case_counts = operational_data.groupby(['State', 'Classification']).size().unstack(fill_value=0).reindex(activity_state_order)

state_activity_findings = pd.DataFrame({
    'Services': activity_state_sizes,
    'MostCommonActivity': state_activity_percent.idxmax(axis=1),
    'TopActivityPct': state_activity_percent.max(axis=1),
    'MultipleActivityPct': state_multiple_activity_pct
})
state_activity_findings['LargestLocationGapActivity'] = state_activity_location_gap.abs().idxmax(axis=1)
state_activity_findings['RuralMinusUrbanGapPP'] = [
    state_activity_location_gap.loc[state, state_activity_findings.loc[state, 'LargestLocationGapActivity']]
    if state in activity_state_order else np.nan
    for state in state_activity_findings.index
]
display(state_activity_findings.round(1))


In [ ]:
# One visual brings together the main findings from all state cases.
fig = plt.figure(figsize=(19, 12))
grid = fig.add_gridspec(2, 2, width_ratios=[3.2, 1.4], hspace=0.46, wspace=0.35)
mix_ax = fig.add_subplot(grid[0, 0])
gap_ax = fig.add_subplot(grid[1, 0])
multi_ax = fig.add_subplot(grid[:, 1])
row_labels = [f'{state} (n={activity_state_sizes[state]:,})' for state in state_activity_percent.index]
mix_image = mix_ax.imshow(state_activity_percent.astype(float), cmap='Blues', vmin=0, vmax=100, aspect='auto')
mix_ax.set_yticks(range(len(row_labels)), row_labels)
mix_ax.set_xticks(range(len(activity_cols)), activity_short_labels, rotation=25, ha='right', fontsize=9)
mix_ax.set_title('A. Services Providing Each Activity (%)', loc='left', pad=12)
for row in range(len(state_activity_percent)):
    for col in range(len(activity_cols)):
        value = float(state_activity_percent.iloc[row, col])
        mix_ax.text(col, row, f'{value:.1f}', ha='center', va='center', fontsize=9,
                    color='white' if value >= 55 else 'black')
fig.colorbar(mix_image, ax=mix_ax, pad=0.02, label='% of services')

gap_limit = max(1, np.nanmax(np.abs(state_activity_location_gap.to_numpy(dtype=float))))
gap_image = gap_ax.imshow(state_activity_location_gap.astype(float), cmap=LinearSegmentedColormap.from_list('location_gap', ['tab:orange', 'white', 'tab:blue']),
                          vmin=-gap_limit, vmax=gap_limit, aspect='auto')
gap_ax.set_yticks(range(len(activity_state_order)), [
    f'{state} (U={location_case_counts.loc[state, "Urban"]:,}; R={location_case_counts.loc[state, "Rural"]:,})'
    for state in activity_state_order
])
gap_ax.set_xticks(range(len(activity_cols)), activity_short_labels, rotation=25, ha='right', fontsize=9)
gap_ax.set_title('B. Rural Minus Urban Activity Share (percentage points)', loc='left', pad=12)
for row in range(len(activity_state_order)):
    for col in range(len(activity_cols)):
        value = float(state_activity_location_gap.iloc[row, col])
        gap_ax.text(col, row, f'{value:+.1f}' if pd.notna(value) else 'N/A', ha='center', va='center', fontsize=9,
                    color='white' if abs(value) > gap_limit * .6 else 'black')
fig.colorbar(gap_image, ax=gap_ax, pad=0.02, label='Rural - Urban (pp)')

multi_ax.barh(state_multiple_activity_pct.index, state_multiple_activity_pct.values,
              color=['tab:blue'] * len(activity_state_order) + ['tab:orange'], alpha=.7)
multi_ax.invert_yaxis()
for row, value in enumerate(state_multiple_activity_pct):
    multi_ax.text(value + 1, row, f'{value:.1f}%', va='center', fontsize=10)
multi_ax.axvline(state_multiple_activity_pct['Australia'], color='tab:orange', linestyle='--', linewidth=1)
multi_ax.set_xlim(0, max(60, state_multiple_activity_pct.max() + 12))
multi_ax.set_title('C. Services Providing\nTwo or More Activities', loc='left', pad=12)
multi_ax.set_xlabel('% of services')
multi_ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Service Activity Findings by State and Territory', fontsize=17, y=.98)
fig.text(.5, .025, 'Source: current operational_data sample. Activity groups overlap; Australia is weighted by service records.\n'
         'Unmatched SOS records are grouped as Rural (Section 4.4). Small Rural samples (especially ACT) require cautious interpretation.',
         ha='center', fontsize=10)
fig.subplots_adjust(left=.17, right=.97, top=.92, bottom=.12)
plt.show()


#### 9.1.11 Supplementary Activity Combinations: End of 2024

The following existing comparison and Section 9.2 use approvals through 2024. The state cases and Australia case above use the full `operational_data` sample.


In [ ]:
# Join supporting fields to a new analysis table; leave operational_data unchanged.
operations = operational_data.merge(
    education_raw[['ServiceApprovalNumber', 'ServiceType', 'NumberOfApprovedPlaces',
                   'ServiceApprovalGrantedDate', 'OverallRating']],
    on='ServiceApprovalNumber', how='left', validate='one_to_one'
)
operations[activity_cols] = operations[activity_cols].replace({1: 'Yes', 0: 'No'})
operations['ApprovalDate'] = pd.to_datetime(
    operations['ServiceApprovalGrantedDate'], format='%d/%m/%Y', errors='coerce'
)
operations = operations.loc[operations['ApprovalDate'] <= '2024-12-31'].copy()
operations['ApprovedPlaces'] = pd.to_numeric(operations['NumberOfApprovedPlaces'], errors='coerce')
activity_labels = {
    'Long Day Care': 'Long Day Care',
    'Preschool/Kindergarten - Part of a School': 'Preschool - School',
    'Preschool/Kindergarten - Stand alone': 'Preschool - Standalone',
    'Outside school Hours Care - Before School': 'OSHC - Before School',
    'Outside school Hours Care - After School': 'OSHC - After School',
    'Outside school Hours Care - Vacation Care': 'OSHC - Vacation Care',
    'Other': 'Other'
}
activity_flags = operations[list(activity_labels)].apply(lambda x: x.astype('string').str.strip())
activity_known = activity_flags.isin(['Yes', 'No']).all(axis=1)
activity_yes = activity_flags.eq('Yes').fillna(False)
operations['ActivityCount'] = activity_yes.sum(axis=1).where(activity_known)
operations['ActivityCombination'] = activity_yes.rename(columns=activity_labels).apply(
    lambda row: ' + '.join(row.index[row]) or 'No activity flagged', axis=1
).where(activity_known, 'Missing / unrecognised activity flag')
activity_profile = pd.DataFrame({
    'Services': activity_yes.sum(),
    'UnknownFlags': (~activity_flags.isin(['Yes', 'No'])).sum()
}).rename(index=activity_labels)
activity_profile['ServicePct'] = activity_profile['Services'] / len(operations) * 100
activity_count_profile = operations['ActivityCount'].value_counts(dropna=False).sort_index().to_frame('Services')
activity_combinations = operations['ActivityCombination'].value_counts().to_frame('Services')
activity_combinations['ServicePct'] = activity_combinations['Services'] / len(operations) * 100
multiple_activity_n = operations['ActivityCount'].ge(2).sum()
print(f'Analysis sample: {len(operations):,} services. Multiple activities: {multiple_activity_n:,} '
      f'({multiple_activity_n / len(operations):.1%} of all services).')
display(activity_profile.round(2))
display(activity_count_profile)
display(activity_combinations.head(10).round(2))


In [ ]:
# AI
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
activity_plot = activity_profile.sort_values('Services')
axes[0].barh(activity_plot.index, activity_plot['Services'], alpha=0.7)
for y, (_, row) in enumerate(activity_plot.iterrows()):
    axes[0].text(row['Services'] + 70, y, f"{row['Services']:,.0f} ({row['ServicePct']:.1f}%)", va='center', fontsize=9)
axes[0].set_xlim(0, activity_plot['Services'].max() * 1.35)
axes[0].set(title='Services Providing Each Activity', xlabel='Number of services')
combination_plot = activity_combinations.head(10).iloc[::-1]
# Line breaks retain the full names of long combinations.
combination_names = [name.replace(' + ', ' +\n') for name in combination_plot.index]
axes[1].barh(combination_names, combination_plot['Services'], alpha=0.7)
axes[1].set(title='Ten Most Common Activity Combinations', xlabel='Number of services')
axes[1].tick_params(axis='y', labelsize=8)
fig.suptitle('Activity Flags Overlap; Exact Combinations Are Mutually Exclusive')
plt.tight_layout()
plt.show()


### 9.2 Capacity by Service Activity

**How does capacity differ across services providing each activity?** Activity groups overlap: the same service can appear in several boxes. This is not a mutually exclusive service-type comparison. Use observed positive approved places for Centre-Based Care (CBC); Family Day Care capacity is almost entirely missing and is not pooled with CBC. No capacity is imputed in this comparison. Tables report available capacity relative to all CBC providers of each activity.

Boxplots show median and IQR with whiskers at 1.5 × IQR; all outlier points remain visible. Approved places are not actual attendance or vacancies. State, Urban/Rural, service-type and children-per-service capacity comparisons are not repeated here.


#### 9.2.1 Shared Capacity Fields and Interactive Controls

Both boxplots use `education_sos` directly. Add observed `ApprovedPlaces` and the seven-flag `ActivityCount` once on that table; use `capacity_analysis_mask` to select the existing end-of-2024 CBC sample. No chart-specific service DataFrame is created.

Select a state/territory or Australia in each chart. Hover anywhere inside a box or around its whiskers to see n, minimum, Q1, median, Q3 and maximum in one upright tooltip. Outlier hover also shows the individual capacity. The transparent hover area is wider than the visible box, so groups with no outliers, a single service or equal values remain interactive. Empty groups show n=0 without a box; hover near zero at their category position for a no-data message.


In [ ]:
# Keep one service-level table: add the two shared fields to education_sos.
# ApprovedPlaces uses observed source values, NOT ApprovedPlaces_Clean imputation.
education_sos['ApprovedPlaces'] = pd.to_numeric(
    education_sos['NumberOfApprovedPlaces'], errors='coerce'
)
education_sos['ActivityCount'] = education_sos[activity_cols].eq('Yes').sum(axis=1).where(
    education_sos[activity_cols].isin(['Yes', 'No']).all(axis=1)
)

# A Boolean mask selects the original end-of-2024 CBC sample; it is not a new table.
capacity_analysis_mask = (
    education_sos['ServiceType'].eq('Centre-Based Care')
    & education_sos['ApprovedPlaces'].gt(0)
    & education_sos['ActivityCount'].notna()
    & pd.to_datetime(
        education_sos['ServiceApprovalGrantedDate'], format='%d/%m/%Y', errors='coerce'
    ).le('2024-12-31')
)
capacity_regions = ['Australia', 'NSW', 'VIC', 'QLD', 'WA', 'SA', 'TAS', 'ACT', 'NT']


def make_capacity_boxplot(group_by):
    """Build either boxplot directly from education_sos, with the same state filter."""
    by_activity = group_by == 'activity'
    if by_activity:
        groups = list(activity_labels)
        group_labels = list(activity_labels.values())
        chart_title = 'CBC Capacity by Activity Provided'
    else:
        groups = sorted(education_sos.loc[capacity_analysis_mask, 'ActivityCount'].astype(int).unique())
        group_labels = [str(count) for count in groups]
        chart_title = 'CBC Capacity by Number of Activities Provided'

    figure = go.Figure()
    state_chart_settings = []
    for region in capacity_regions:
        state_mask = capacity_analysis_mask.copy()
        if region != 'Australia':
            state_mask &= education_sos['State'].eq(region)
        tick_labels, median_lines = [], []
        # Padding gives even a single-value box a usable hover area.
        region_max = education_sos.loc[state_mask, 'ApprovedPlaces'].max()
        hover_padding = max(5.0, float(region_max) * 0.015) if pd.notna(region_max) else 5.0

        for position, group in enumerate(groups):
            if by_activity:
                group_mask = education_sos[group].eq('Yes')
            else:
                group_mask = education_sos['ActivityCount'].eq(group)
            # Select only the capacity Series needed by this box, not a copied DataFrame.
            values = education_sos.loc[state_mask & group_mask, 'ApprovedPlaces']
            n = len(values)
            label = group_labels[position]
            tick_labels.append(f'{label} (n={n:,})' if by_activity else f'{label}<br>(n={n:,})')
            if n:
                q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
                # Explicit quartiles and whiskers match the original Matplotlib boxes.
                # Plotly's default raw-sample quartiles can differ in small groups.
                iqr = q3 - q1
                lower_whisker = min(q1, values.loc[values >= q1 - 1.5 * iqr].min())
                upper_whisker = max(q3, values.loc[values <= q3 + 1.5 * iqr].max())
                hover = (
                    f'<b>{label} | {region}</b><br>Services: {n:,}<br>'
                    f'Min: {values.min():.1f}<br>Q1: {q1:.1f}<br>'
                    f'Median: {median:.1f}<br>Q3: {q3:.1f}<br>Max: {values.max():.1f}'
                )
                # Draw an orange median line over the standard black/white box.
                if by_activity:
                    median_lines.append(dict(type='line', x0=median, x1=median,
                        y0=position - 0.25, y1=position + 0.25, line=dict(color=plotly_orange, width=2)))
                else:
                    median_lines.append(dict(type='line', y0=median, y1=median,
                        x0=position - 0.25, x1=position + 0.25, line=dict(color=plotly_orange, width=2)))
            else:
                hover = f'<b>{label} | {region}</b><br>No services with available capacity'

            figure.add_trace(go.Box(
                # One nested sample array per precomputed box preserves outlier points.
                x=([values.tolist()] if by_activity else [position]) if n else [],
                y=([position] if by_activity else [values.tolist()]) if n else [],
                q1=[q1] if n else [], median=[median] if n else [], q3=[q3] if n else [],
                lowerfence=[lower_whisker] if n else [], upperfence=[upper_whisker] if n else [],
                orientation='h' if by_activity else 'v',
                width=0.5, name=label, showlegend=False,
                line=dict(color='black', width=1), fillcolor='white',
                marker=dict(color='gray', size=3, opacity=0.4),
                quartilemethod='linear', boxpoints='outliers', jitter=0, pointpos=0,
                # Hide Plotly's angled box-statistic labels; keep the custom point tooltip.
                hoveron='points',
                hovertemplate=hover + ('<br>Capacity: %{x:.1f}' if by_activity else '<br>Capacity: %{y:.1f}') + '<extra></extra>',
                visible=region == 'Australia'
            ))
            # A transparent rectangle handles hover across the whole box and whiskers.
            # It changes no observations, quartiles, whiskers, colours or visible marks.
            # Scalar text is required for Plotly fill hover; hovertemplate is ignored here.
            hover_low = max(0.0, lower_whisker - hover_padding) if n else 0.0
            hover_high = upper_whisker + hover_padding if n else hover_padding * 2
            group_edges = [position - 0.35, position + 0.35,
                           position + 0.35, position - 0.35, position - 0.35]
            capacity_edges = [hover_low, hover_low, hover_high, hover_high, hover_low]
            figure.add_trace(go.Scatter(
                x=capacity_edges if by_activity else group_edges,
                y=group_edges if by_activity else capacity_edges,
                mode='lines', line=dict(width=0, color='black'),
                fill='toself', fillcolor='rgba(0, 0, 0, 0)',
                hoveron='fills', hoverinfo='text', text=hover,
                name=label, showlegend=False,
                visible=region == 'Australia'
            ))
        state_chart_settings.append((tick_labels, median_lines, int(state_mask.sum())))

    buttons = []
    category_axis = 'yaxis' if by_activity else 'xaxis'
    capacity_axis = 'xaxis' if by_activity else 'yaxis'
    for state_number, region in enumerate(capacity_regions):
        tick_labels, median_lines, n = state_chart_settings[state_number]
        buttons.append(dict(label=region, method='update', args=[
            {'visible': [trace // (2 * len(groups)) == state_number for trace in range(len(figure.data))]},
            {f'{category_axis}.ticktext': tick_labels, f'{capacity_axis}.autorange': True,
             'shapes': median_lines,
             'title.text': f'{chart_title} — {region} (n={n:,})'}
        ]))

    labels, median_lines, national_n = state_chart_settings[0]
    figure.update_layout(
        template='plotly_white', width=1000, height=600,
        title=dict(text=f'{chart_title} — Australia (n={national_n:,})', y=0.97, x=0.05),
        margin=dict(l=240 if by_activity else 90, r=40, t=140, b=110),
        updatemenus=[dict(buttons=buttons, active=0, x=0.08, y=1.16, xanchor='left')],
        annotations=[dict(text='Region', x=0, y=1.14, xref='paper', yref='paper',
                          showarrow=False, xanchor='left')],
        shapes=median_lines,
        hovermode='closest', hoverdistance=30,
        hoverlabel=dict(bgcolor='white', bordercolor='black', font_color='black', align='left')
    )
    figure.update_layout(**{
        category_axis: dict(tickmode='array', tickvals=list(range(len(groups))), ticktext=labels,
                            range=[-0.65, len(groups) - 0.35],
                            title='' if by_activity else 'Number of Activities Provided', showgrid=False),
        capacity_axis: dict(title='Approved Places', rangemode='tozero', showgrid=False)
    })
    return figure




#### 9.2.2 Interactive Capacity Distribution by Activity


In [ ]:
# Reuse the common education_sos fields and Boolean mask above.
fig_capacity_activity = make_capacity_boxplot('activity')
fig_capacity_activity.show()


In [ ]:
education_sos['NumberOfApprovedPlaces'].describe()

### 9.3 Capacity by Number of Activities Provided

Compare capacity across services offering different numbers of activities. Select directly from `education_sos` with `capacity_analysis_mask`, using the same sample as the preceding activity boxplot: Centre-Based Care services approved by the end of 2024, with observed positive approved places and complete activity flags. Count the seven existing Yes/No activity fields separately; each service appears in one group. Family Day Care is excluded under the existing capacity-analysis rule, and no capacity is imputed.

Boxes show the median and interquartile range, with whiskers at 1.5 times the IQR and all outliers visible. Labels give the number of services with available capacity in each group.


#### 9.3.1 Interactive Capacity Distribution by Activity Count


In [ ]:
# Same source table and sample as the activity boxplot; only the grouping changes.
fig_capacity_count = make_capacity_boxplot('count')
fig_capacity_count.show()
